# Job Queue

> Resource-aware job queue for sequential plugin execution with cancellation support

In [ ]:
#| default_exp core.queue

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import asyncio
import heapq
import uuid
from collections import defaultdict
from dataclasses import dataclass, field, asdict
from datetime import datetime, timezone
from enum import Enum
from pathlib import Path
from typing import (
    Any, AsyncIterator, Callable, Dict, List, Optional, Protocol,
    Set, Tuple, TYPE_CHECKING, runtime_checkable,
)

from cjm_plugin_system.core.errors import (
    JobError, TracebackPolicy, map_bare_exception_to_job_error,
)
from cjm_plugin_system.core.ports import (
    Composition, CompositionBindingError, CompositionRun, NodeState,
    new_composition_run, resolve_node_kwargs,
)
from cjm_plugin_system.core.journal_store import (
    JournalEvent, JournalStore, LIVENESS_EVENT_TYPES, SubstrateEventType,
)
from cjm_plugin_system.core.diagnostics_store import DiagnosticRecord, DiagnosticsStore
from cjm_plugin_system.core.wire import CallEnvelope, reset_call_envelope, set_call_envelope
from cjm_plugin_system.core._telemetry import attribute_gpu_to_worker_subtree

# CR-6: CapabilityManager continues to satisfy JobQueueDependencies structurally.
# Imported under TYPE_CHECKING to thin runtime coupling per OQ-5 resolution —
# JobQueue stays inline in cjm-plugin-system for v1; the Protocol enables clean
# future extraction.
if TYPE_CHECKING:
    from cjm_plugin_system.core.manager import CapabilityManager

# SG-39: library modules use `logging.getLogger(__name__)` and let the host
# (CLI entry point, FastHTML app, worker subprocess) own `basicConfig`.
import logging

The `JobQueue` provides a resource-aware **multi-lane** job queue for
capability execution (stage 3 / CR-16 rework):

```
┌──────────────────────────────────────────────────────────────┐
│                    User Application                          │
│  ┌──────────────────────────────────────────────────────┐    │
│  │                    JobQueue                          │    │
│  │  ┌───────────┐  ┌─────────────────┐  ┌────────────┐  │    │
│  │  │ Pending   │  │ Running Jobs    │  │ History    │  │    │
│  │  │ Jobs      │→ │ (lanes; gated   │→ │ (completed/│  │    │
│  │  │ (heap)    │  │  by admission)  │  │ cancelled) │  │    │
│  │  └───────────┘  └─────────────────┘  └────────────┘  │    │
│  │        │                 ▲                           │    │
│  │        ▼                 │ lazy member Jobs          │    │
│  │  ┌─────────────────────────────────┐                 │    │
│  │  │ Compositions (DAGs w/ bindings) │                 │    │
│  │  └─────────────────────────────────┘                 │    │
│  └──────────────────────────────────────────────────────┘    │
│         │                │                                   │
│         ▼                ▼                                   │
│  ┌──────────────┐  ┌─────────────┐                           │
│  │ Empirical    │  │ Plugin      │                           │
│  │ store+sysmon │  │ Manager     │                           │
│  └──────────────┘  └─────────────┘                           │
└──────────────────────────────────────────────────────────────┘
```

Key features:

- **Priority queue**: Higher priority jobs execute first (FIFO within same priority)
- **Multi-lane dispatch (stage 3)**: independent jobs run concurrently when the
  admission ladder allows — derived from empirical resource records + live
  sysmon telemetry, never from declared metadata; no evidence → exclusive
  (the first run is the measurement run)
- **Compositions (stage 3 / CR-16)**: DAGs of capability invocations with
  execution-time-bound inputs (`OutputRef` markers); member Jobs created
  lazily as dependencies complete — replaces `submit_sequence`
- **Cancellation**: Cancel pending or running jobs with graceful fallback to force termination
- **Progress tracking**: Poll progress and status messages during execution
- **Observable**: push-based event bus + queue state for UI integration

## Enums

`JobStatus` captures the lifecycle a job moves through. `JobEventType` enumerates the push-based events emitted on the CR-6 multi-subscriber event bus. `CancelPhase` surfaces the substrate's cancel state machine (cooperative → force → reloading → completed) so UIs can render cancel progress.

In [ ]:
#| export
class JobStatus(str, Enum):
    """Status of a job in the queue."""
    pending = "pending"
    running = "running"
    completed = "completed"
    failed = "failed"
    cancelled = "cancelled"


class JobEventType(str, Enum):
    """Push-based job event types (CR-6; stage-3 composition rework; CR-14
    journal-primary emission).

    Emitted by JobQueue through the single emission path (`_publish_event`):
    journal-class events become durable journal rows AND fan out to live
    subscribers; liveness-class events (`LIVENESS_EVENT_TYPES` in
    `core.journal_store`) fan out only — their final values ride the
    terminal STATE_TRANSITION row. Consumers subscribe via
    `queue.events(job_id)` / `queue.events_for_composition(comp_id)` /
    `queue.all_events()` and receive `JobEvent` instances asynchronously.

    COMPOSITION_ADVANCED replaced the retired SEQUENCE_ADVANCED at execution
    stage 3 (CR-16: compositions replace sequences outright): it fires when a
    member job's completion unlocks downstream composition nodes — payload
    carries the completed node id + the newly enqueued node ids.

    The reserved-never-emitted LOG_APPENDED was RETIRED at stage 7 (CR-14):
    log-follow is a diagnostics-store cursor read (`get_job_diagnostics`),
    not a push event — there are no log files or byte offsets anymore.
    Non-job substrate events (worker lifecycle, config, runs) live in
    `core.journal_store.SubstrateEventType`.
    """
    STATE_TRANSITION = "state_transition"          # JobStatus changed (Stage 1); terminal transitions carry the job snapshot (CR-14)
    PROGRESS_CHANGED = "progress_changed"          # progress / status_message updated (Stage 1; liveness — never journaled)
    COMPOSITION_ADVANCED = "composition_advanced"  # Composition enqueued downstream nodes (stage 3)
    RESOURCE_SNAPSHOT = "resource_snapshot"        # Periodic worker + sysmon stats (Stage 3; liveness — never journaled)
    BLOCK_REASON_CHANGED = "block_reason_changed"  # Scheduler block reason changed (Stage 4)
    CANCEL_PHASE_CHANGED = "cancel_phase_changed"  # Cancel state machine progressed (Stage 4)
    RETRY_STARTED = "retry_started"                # Reactive retry beginning (CR-7 + Stage 4)


class CancelPhase(str, Enum):
    """Phase of a cancellation in progress (CR-6 + CR-4 pairing).

    Surfaces the substrate's cancel state machine. Stage 4 wires the
    transitions; Stage 1 reserves the enum so `Job.cancel_phase` can be
    typed correctly without dangling forward references.
    """
    COOPERATIVE = "cooperative"  # Cancel signal sent; awaiting plugin acknowledgement
    FORCE = "force"              # Cooperative timeout; force-terminating worker
    RELOADING = "reloading"      # Worker reload in progress post-force-kill
    COMPLETED = "completed"      # Cancellation fully complete

## JobQueueDependencies

The CR-6 substrate dependency Protocol that JobQueue requires. `CapabilityManager` satisfies this structurally, so existing hosts continue to work — passing a `CapabilityManager` to `JobQueue(deps=...)` still type-checks. The Protocol enables clean future extraction of JobQueue into a separate library per OQ-5 resolution. The audit's locked design listed 3 methods; investigation surfaced 2 more (`get_capability`, `reload_capability`, `get_plugin_logs`) that are consumed in the current execute path. All 5 are included so the Protocol genuinely captures the surface the queue uses.

In [ ]:
#| export
@runtime_checkable
class JobQueueDependencies(Protocol):
    """Substrate dependencies the JobQueue requires (CR-6 + stage 3).

    CapabilityManager satisfies this structurally; the Protocol exists so JobQueue
    can be tested in isolation (with a lightweight test double) and so a future
    extraction into a separate library has no API constraint locked in.

    The first 4 methods are the CR-6 execute-path surface (CR-14 retired
    `get_plugin_logs` — log retrieval is a diagnostics-store query now). The
    stage-3 additions (CR-16 multi-lane admission) are consumed DEFENSIVELY
    via getattr — a deps implementation without them yields no admission
    evidence, so every job runs exclusive = exact pre-stage-3 single-lane
    behavior. Older test doubles keep working unchanged. CR-14 also reads
    `journal_store` / `diagnostics_store` ATTRIBUTES via getattr when the
    queue isn't constructed with explicit stores — a deps without them
    (test doubles) simply yields no journaling.
    """
    def get_capability_meta(self, name_or_id: str) -> Optional[Any]: ...
    def get_capability(self, name_or_id: str) -> Optional[Any]: ...
    async def execute_capability_async(self, name_or_id: str, *args: Any, **kwargs: Any) -> Any: ...
    def reload_capability(self, name_or_id: str) -> Any: ...
    # Stage 3 (CR-16) admission surface:
    def get_admission_profile(self, name_or_id: str) -> Optional[Dict[str, Any]]: ...
    def get_instance_concurrency_cap(self, name_or_id: str) -> Optional[int]: ...
    async def get_global_stats(self) -> Dict[str, Any]: ...
    # Stage 4 (CR-17 pt 2) task channel — invoked only for task-addressed jobs
    # (Job.task_name set); execute-channel jobs never touch it, so older test
    # doubles keep working unchanged.
    async def execute_capability_task_async(self, name_or_id: str, task_name: str, method: str, **kwargs: Any) -> Any: ...

## Job / JobEvent / QueueStats / _Subscription

`Job` is the queued execution unit (CR-6 reshape: `capability_instance_id` per CR-10, `JobError` typed error per CR-5, datetime timestamps; stage 3 replaced the sequence tags with composition tags — `composition_id`/`node_id` — alongside the cancel-phase + block-reason + retry fields). `JobEvent` is the push-bus payload — every event carries full tag context (`job_id`, optional `composition_id`/`node_id`, `capability_instance_id`) so consumers subscribed to any filter view get the same shape. `QueueStats` is the aggregate-counts shape returned by `get_stats()`. `_Subscription` is the internal per-subscriber bounded-queue + drop-counter holder.

`Job` keeps backward-compat `@property` aliases for `capability_name` (returns `capability_instance_id`) and `created_at` (returns `submitted_at.timestamp()`), tagged for SG-48 removal after the consumer cascade migrates.

In [ ]:
#| export
@dataclass
class Job:
    """A queued plugin execution request (CR-6 reshape; stage-3 composition
    rework renamed the sequence tags to composition tags).

    `composition_id` / `node_id` are set when the job is a lazily-created
    member of a composition (CR-16) — they ride every JobEvent so
    `events_for_composition` subscribers see member lifecycle events.

    `run_id` / `actor` (CR-14 follow-up) are host-tier correlation tags:
    cores pass their run-manifest id + initiating actor at submit so every
    journal row for this job links back to the run record (run manifest ↔
    journal linkage) and carries who/what initiated it.
    """
    id: str  # Unique job identifier (UUID)
    capability_instance_id: str  # Target plugin instance (per CR-10)
    args: Tuple[Any, ...]  # Positional arguments for execute()
    kwargs: Dict[str, Any]  # Keyword arguments for execute()
    status: JobStatus = JobStatus.pending  # Current job status
    priority: int = 0  # Higher = more urgent
    submitted_at: datetime = field(default_factory=lambda: datetime.now(timezone.utc))  # When submitted
    started_at: Optional[datetime] = None  # When execution started
    completed_at: Optional[datetime] = None  # When execution finished
    progress: float = 0.0  # 0.0 to 1.0, or -1.0 for indeterminate
    status_message: str = ""  # Descriptive status message
    result: Any = None  # Execution result (if completed)
    error: Optional[JobError] = None  # Structured failure summary (CR-5)
    composition_id: Optional[str] = None  # Set when part of a composition (stage 3)
    node_id: Optional[str] = None  # Composition node this job executes (stage 3)
    task_name: Optional[str] = None  # Task-channel address: adapter task (stage 4, CR-17 pt 2)
    method: Optional[str] = None  # Task-channel address: adapter method (stage 4)
    run_id: Optional[str] = None  # Host-tier run correlation (CR-14 follow-up; core run manifests)
    actor: Optional[str] = None  # Who/what initiated the work (CR-14 follow-up)
    control: Dict[str, Any] = field(default_factory=dict)  # Per-call control flags (force/cache-bypass — CR-15 cat 4); rides CallEnvelope.control
    cancel_requested_at: Optional[datetime] = None  # When cancel was requested (Stage 4)
    cancel_phase: Optional[CancelPhase] = None  # Active cancel phase (Stage 4)
    block_reason: Optional[str] = None  # Why the scheduler is blocking (Stage 4)
    retry_count: int = 0  # Reactive retries attempted (CR-7 + Stage 4)
    last_resource_snapshot: Optional[Any] = None  # Stage 3 wires this (ResourceSnapshot)

    def __lt__(self, other: 'Job') -> bool:
        """Compare jobs for priority queue (higher priority first, then FIFO).

        Negate priority so higher values come first in the min-heap; use
        submitted_at as tiebreaker (earlier first). datetime supports < natively.
        """
        return (-self.priority, self.submitted_at) < (-other.priority, other.submitted_at)

    # REMOVE-AFTER-OVERHAUL: backward-compat aliases for the CR-6 rename
    # cascade. External consumers (job-monitor library, decomp host) read
    # `job.capability_name` and `job.created_at` today; these properties keep
    # those call sites working until SG-47/SG-48 sweeps migrate them.
    @property
    def capability_name(self) -> str:
        return self.capability_instance_id

    @property
    def created_at(self) -> float:
        return self.submitted_at.timestamp()


@dataclass
class JobEvent:
    """A push-based job event (CR-6; stage-3 composition tags).

    Carries full tag context so a subscriber to `all_events()`, `events(job_id)`,
    or `events_for_composition(comp_id)` receives identically-shaped instances.
    `payload` is a per-event-type structured dict (e.g., STATE_TRANSITION carries
    `{"from": "pending", "to": "running"}`; COMPOSITION_ADVANCED carries
    `{"completed_node": ..., "enqueued_nodes": [...]}`).

    `run_id` / `actor` (CR-14 follow-up) ride from the Job so journal rows
    written by the single emission path carry the host-tier correlation.
    """
    type: JobEventType
    job_id: str
    capability_instance_id: str
    composition_id: Optional[str] = None
    node_id: Optional[str] = None
    run_id: Optional[str] = None  # Host-tier run correlation (CR-14 follow-up)
    actor: Optional[str] = None  # Who/what initiated (CR-14 follow-up)
    timestamp: datetime = field(default_factory=lambda: datetime.now(timezone.utc))
    payload: Dict[str, Any] = field(default_factory=dict)


@dataclass
class QueueStats:
    """Aggregate counts returned by `JobQueue.get_stats()` (CR-6)."""
    total_pending: int
    total_completed: int
    total_failed: int
    total_cancelled: int


@dataclass
class _Subscription:
    """Internal: per-subscriber bounded queue + drop counter (CR-6 event bus).

    Slow subscribers backpressure themselves via `asyncio.QueueFull` drop;
    the publisher never blocks. `dropped_count` surfaces visibility for
    operators / future telemetry.
    """
    queue: 'asyncio.Queue[JobEvent]'
    dropped_count: int = 0

## ResourceSnapshot

Point-in-time worker + GPU stats for one job, populated by sampling the worker proxy's `get_stats()` and (optionally) the configured system-monitor plugin's CR-3 typed `get_system_status()` + `list_processes()` methods. Distinct from CR-7's `EmpiricalResourceRecord` which aggregates samples across runs — `ResourceSnapshot` is the substrate's "what's happening right now" surface. Field shape mirrors `cjm-fasthtml-job-monitor`'s existing `ResourceSnapshot` so the cascade migration to the substrate-owned type is structural rather than semantic.

In [ ]:
#| export
@dataclass
class ResourceSnapshot:
    """Point-in-time resource usage for one job (CR-6 Stage 3).

    Worker stats (cpu_percent, memory_rss_mb) come from the plugin proxy's
    `get_stats()`. GPU fields come from the configured system-monitor plugin
    (when set on JobQueue) via CR-3's typed `list_processes()` (per-PID
    matching) and `get_system_status()` (global GPU stats). All GPU fields
    are Optional — None if no sysmon configured or the worker isn't running
    on a GPU.

    Distinct from CR-7's `EmpiricalResourceRecord` (aggregated profile
    across runs); this is "what's happening right now" for one job.
    """
    timestamp: datetime  # When the sample was taken
    worker_pid: int = 0  # OS PID of the plugin worker subprocess
    cpu_percent: float = 0.0  # Worker process CPU%
    memory_rss_mb: float = 0.0  # Worker process resident memory (MB)
    gpu_index: Optional[int] = None  # GPU index the worker is on (None if not GPU-bound)
    gpu_memory_mb: Optional[float] = None  # Worker's GPU memory usage (MB)
    gpu_type: Optional[str] = None  # GPU vendor (NVIDIA / AMD / Intel / None)
    gpu_total_mb: Optional[float] = None  # Total GPU memory available globally (MB)
    gpu_load_percent: Optional[float] = None  # GPU compute utilization (global)

## JobQueue

Main queue class — submission, execution lifecycle, multi-subscriber event bus, observability. CR-6 reshape: takes `deps: JobQueueDependencies` (Protocol) instead of `manager: CapabilityManager` (concrete); adds bounded-queue multi-subscriber event-bus state.

In [ ]:
#| export
class JobQueue:
    """Resource-aware multi-lane job queue with journal-primary observability
    (CR-6 + CR-14; stage-3 CR-16 rework: ready-set dispatch + resource-derived
    admission + composition execution)."""

    # CR-6: per-subscriber bounded queue size. Slow subscribers backpressure
    # themselves via QueueFull drop; publisher never blocks. 100 events is
    # large enough that a UI tab pause of a few seconds doesn't drop events
    # under typical workloads, small enough that a stuck subscriber doesn't
    # accumulate unbounded memory.
    SUBSCRIBER_QUEUE_MAXSIZE: int = 100

    def __init__(
        self,
        deps: JobQueueDependencies,        # Substrate dependencies (CapabilityManager satisfies structurally)
        max_history: int = 100,            # Max completed jobs to retain
        cancel_timeout: float = 3.0,       # Seconds to wait for cooperative cancel
        progress_poll_interval: float = 1.0,  # Seconds between progress polls
        sysmon_capability_name: Optional[str] = None,  # CR-3 MonitorPlugin instance for GPU stats (None = no GPU info)
        resource_snapshot_cadence_polls: int = 4,  # Sample resources every Nth progress poll
        max_concurrent_lanes: int = 4,     # Stage 3: max in-flight jobs (admission still gates each)
        gpu_headroom_fraction: float = 0.9,  # Stage 3: blunt GPU admission margin (budget = total * fraction)
        journal: Optional[JournalStore] = None,  # CR-14: journal sink; defaults to deps.journal_store (the manager's)
        diagnostics: Optional[DiagnosticsStore] = None,  # CR-14: diagnostics sink for get_job_diagnostics; defaults to deps.diagnostics_store
    ):
        """Initialize the job queue.

        Stage 3 (CR-16): the queue dispatches multiple admissible jobs
        concurrently (`max_concurrent_lanes` is the operator safety valve);
        per-job admission derives from empirical resource records + live
        sysmon telemetry — see `_pop_next_admissible`. Worst case (no
        records, no sysmon) every job runs exclusive, which is exactly the
        pre-stage-3 single-lane behavior.

        CR-14 (stage 7): emission is journal-primary — `_publish_event`
        writes journal-class events as durable rows before fanning out to
        live subscribers. The stores default to the deps' (CapabilityManager's)
        stores via getattr, so the cores gain journaling with zero host
        changes; a deps without them (test doubles) yields no journaling.
        """
        self._deps = deps
        self.max_history = max_history
        self.cancel_timeout = cancel_timeout
        self.progress_poll_interval = progress_poll_interval
        self._sysmon_name = sysmon_capability_name
        self.resource_snapshot_cadence_polls = max(1, resource_snapshot_cadence_polls)
        self.max_concurrent_lanes = max(1, max_concurrent_lanes)
        self.gpu_headroom_fraction = gpu_headroom_fraction
        self.logger = logging.getLogger(f"{__name__}.{type(self).__name__}")

        # CR-14: observability stores (journal-primary emission). The wedge
        # flag is the named-tension resolution: an in-flight journal-append
        # failure logs at ERROR and wedges the queue — the NEXT submit
        # refuses loudly (never silently drop the audit trail; never corrupt
        # in-flight lane state by raising mid-finalization).
        self._journal: Optional[JournalStore] = (
            journal if journal is not None else getattr(deps, 'journal_store', None))
        self._diagnostics: Optional[DiagnosticsStore] = (
            diagnostics if diagnostics is not None else getattr(deps, 'diagnostics_store', None))
        self._journal_wedged = False

        # CR-14 follow-up: queue-scoped run context. A host whose queue
        # lifetime IS one run (the cores' CLI shape) sets this once via
        # `set_run_context`; submit-time `run_id=`/`actor=` and
        # `Composition.run_id/actor` override per call. Long-lived-queue
        # hosts (GUIs) leave it unset and pass per-submit tags instead.
        self.default_run_id: Optional[str] = None
        self.default_actor: Optional[str] = None

        # State
        self._pending: List[Job] = []  # Priority heap
        self._running: Dict[str, Job] = {}  # In-flight jobs by id (stage 3: multi-lane)
        self._running_tasks: Dict[str, asyncio.Task] = {}  # Per-job execute tasks
        self._history: List[Job] = []  # Completed/cancelled/failed jobs
        self._jobs: Dict[str, Job] = {}  # All jobs by ID for fast lookup

        # Stage 3: composition registry (replaces the CR-6 sequence registry).
        # CompositionRun instances keyed by run id. Currently unbounded;
        # eviction of terminal runs is a follow-up (parallel to max_history).
        self._compositions: Dict[str, CompositionRun] = {}
        self._composition_completed_events: Dict[str, asyncio.Event] = {}

        # Stage 3 admission ledgers: jobs admitted EXCLUSIVE (no empirical
        # profile — their first run is the measurement run) and the GPU
        # reservation ledger covering admitted-but-not-yet-loaded models.
        self._running_exclusive: Set[str] = set()
        self._gpu_reservations: Dict[str, float] = {}

        # Synchronization
        self._lock = asyncio.Lock()
        self._job_available = asyncio.Event()
        self._job_completed_events: Dict[str, asyncio.Event] = {}

        # CR-6: multi-subscriber event bus.
        # Keys: "all" (firehose), "job:<id>", "comp:<id>". Each subscriber owns
        # one bounded asyncio.Queue; the publisher fans out to every matching
        # key. Independent of `_job_completed_events` (which backs the
        # `wait_for_job` simple-block affordance).
        self._subscribers: Dict[str, List[_Subscription]] = defaultdict(list)

        # Background task
        self._processor_task: Optional[asyncio.Task] = None
        self._running_flag = False
        self._cancel_requested: Set[str] = set()  # Job IDs pending cancellation

    def set_run_context(
        self,
        run_id: Optional[str] = None,  # Host-tier run correlation for subsequent submits
        actor: Optional[str] = None,   # Who/what initiated the run
    ) -> None:
        """Set the queue-scoped default run context (CR-14 follow-up).

        Every subsequent submit without explicit `run_id`/`actor` inherits
        these — the one-queue-per-run CLI cores call this once after
        generating their run-manifest id, and every journal row for the run
        links back to it. Call again (or with None) to change/clear.
        """
        self.default_run_id = run_id
        self.default_actor = actor

    # REMOVE-AFTER-OVERHAUL: backward-compat alias for `self.manager`. The
    # SG-13 regression test + any host inspection still expects a `.manager`
    # attribute. SG-48 sweep retires this after consumer cascade migrates
    # to the new `deps`/`_deps` shape.
    @property
    def manager(self) -> JobQueueDependencies:
        return self._deps

### Job Submission

In [ ]:
#| export
async def _enqueue_job(
    self,
    job: Job,  # Pre-constructed Job (caller fills composition fields if needed)
) -> str:  # job_id
    """Internal: enqueue a pre-constructed Job.

    Caller is responsible for validation (disabled-plugin check, etc.).
    Used by `submit` and by the composition advancement path (stage 3) —
    the latter populates `composition_id` + `node_id` on the Job before
    enqueueing so the member job appears in composition-tagged event streams
    from its first STATE_TRANSITION.
    """
    async with self._lock:
        self._jobs[job.id] = job
        heapq.heappush(self._pending, job)
        self._job_completed_events[job.id] = asyncio.Event()
        self._job_available.set()

    comp_suffix = f" (comp {job.composition_id[:8]} node {job.node_id})" if job.composition_id else ""
    self.logger.info(
        f"Submitted job {job.id[:8]} for {job.capability_instance_id} "
        f"(priority={job.priority}){comp_suffix}"
    )
    return job.id


def _check_journal_wedge(self) -> None:
    """CR-14 wedge gate: refuse new work after a journal-append failure.

    The loud half of the named-tension resolution — a wedged journal must
    never silently drop the audit trail, and the refusal happens at the
    operational boundary (new submissions) rather than mid-finalization
    (which would leak lanes). Clears only by constructing a new queue /
    fixing the journal and resetting `_journal_wedged` deliberately.
    """
    if self._journal_wedged:
        raise RuntimeError(
            "Observability journal is WEDGED (an append failed — see ERROR "
            "logs); refusing new submissions. Fix the journal store and "
            "reset `_journal_wedged` to resume.")


async def submit(
    self,
    capability_instance_id: str,  # Target plugin instance (per CR-10)
    *args,
    priority: int = 0,  # Higher = more urgent
    task: Optional[str] = None,  # Task-channel address: adapter task name (stage 4)
    method: Optional[str] = None,  # Task-channel address: adapter method (set with task)
    run_id: Optional[str] = None,  # Host-tier run correlation (CR-14 follow-up; reserved name, never a plugin kwarg)
    actor: Optional[str] = None,  # Who/what initiated (CR-14 follow-up; reserved name)
    control: Optional[Dict[str, Any]] = None,  # Per-call control flags (force/cache-bypass); reserved name, never a plugin kwarg
    **kwargs
) -> str:  # Returns job_id
    """Submit a job to the queue.

    CR-2: rejects jobs for disabled plugins at submit time (typed
    CapabilityDisabledError) so the failure surface matches CapabilityManager.
    execute_capability's disabled gate. Submitting to a disabled plugin would
    otherwise sit in the queue until execution, then raise — moving the
    check earlier gives operators an actionable signal immediately.

    CR-6: no STATE_TRANSITION event fires at submit because the job is
    already in `pending` state at construction — there's no transition
    to publish. The first STATE_TRANSITION fires when the processor loop
    moves the job pending → running.

    CR-14: refuses loudly when the journal is wedged (see
    `_check_journal_wedge`). `run_id`/`actor` join `priority`/`task`/
    `method` as reserved keyword names (they never reach plugin kwargs):
    cores pass their run-manifest id + initiating actor so every journal
    row for this job carries the host-tier correlation.
    """
    # CR-2: late import to avoid pulling errors module into queue's import
    # path during library load (keeps the worker.py-via-queue.py import
    # chain minimal).
    from cjm_plugin_system.core.errors import CapabilityDisabledError, CapabilityInputError

    self._check_journal_wedge()

    # Stage 4: the task channel addresses adapter+method as a pair.
    if (task is None) != (method is None):
        raise CapabilityInputError(
            f"Task-channel submits require BOTH task and method "
            f"(got task={task!r}, method={method!r})",
            fields_invalid=["task", "method"],
        )

    meta = self._deps.get_capability_meta(capability_instance_id)
    if meta is not None and not meta.enabled:
        raise CapabilityDisabledError(capability_instance_id)

    job = Job(
        id=str(uuid.uuid4()),
        capability_instance_id=capability_instance_id,
        args=args,
        kwargs=kwargs,
        priority=priority,
        task_name=task,
        method=method,
        run_id=run_id if run_id is not None else self.default_run_id,
        actor=actor if actor is not None else self.default_actor,
        control=control or {},
    )
    return await self._enqueue_job(job)

JobQueue._enqueue_job = _enqueue_job
JobQueue._check_journal_wedge = _check_journal_wedge
JobQueue.submit = submit

### Job Control

In [ ]:
#| export
async def cancel(
    self,
    job_id: str  # Job to cancel
) -> bool:  # True if cancelled
    """Cancel a pending or running job.

    CR-6: publishes STATE_TRANSITION when a pending job moves directly to
    cancelled (no transition through running). Running-job cancellation
    publishes CANCEL_PHASE_CHANGED events from `_execute_with_cancellation`.

    Stage 3: when the cancelled job is a composition member,
    `_advance_composition` runs after the lock is released so the cancelled
    status propagates to the composition run (dependents skip; the run
    finalizes when all nodes are terminal). Lock release is required because
    `_advance_composition` may need to enqueue downstream members in some
    flows; asyncio.Lock is not re-entrant.
    """
    advance_target: Optional[Job] = None
    async with self._lock:
        job = self._jobs.get(job_id)
        if not job:
            return False

        if job.status == JobStatus.pending:
            # Move directly to cancelled (job never ran)
            prev_status = job.status
            job.status = JobStatus.cancelled
            job.completed_at = datetime.now(timezone.utc)
            self._pending = [j for j in self._pending if j.id != job_id]
            heapq.heapify(self._pending)
            # SG-13: signal before moving to history so the just-cancelled job's
            # event is set before any eviction could pop it from the dict.
            self._signal_job_completed(job_id)
            self._move_to_history(job)
            # CR-14: route through the centralized emitter so the terminal
            # transition carries the job snapshot (durable history row) —
            # the pre-stage-7 inline publish skipped the snapshot.
            self._emit_state_transition(job, prev_status)
            self.logger.info(f"Cancelled pending job {job_id[:8]}")
            if job.composition_id is not None:
                advance_target = job

        elif job.status == JobStatus.running:
            # Signal cancellation request; phase-transition events (Stage 4)
            # fire from `_execute_with_cancellation` as cooperative → force
            # → reloading → completed unfolds. Composition advance for a
            # running cancelled member happens from `_execute_job`'s
            # finalization path once the cancel resolves.
            self._cancel_requested.add(job_id)
            job.cancel_requested_at = datetime.now(timezone.utc)
            self.logger.info(f"Cancellation requested for running job {job_id[:8]}")
            return True

        else:
            return False  # Already completed/cancelled/failed

    # Outside the lock — safe to call _advance_composition, which may take
    # the lock again via _enqueue_job in other flows.
    if advance_target is not None:
        await self._advance_composition(advance_target)
    return True


def reorder(
    self,
    job_id: str,  # Job to move
    new_priority: int  # New priority value
) -> bool:  # True if reordered
    """Change the priority of a pending job."""
    job = self._jobs.get(job_id)
    if not job or job.status != JobStatus.pending:
        return False

    job.priority = new_priority
    heapq.heapify(self._pending)
    self.logger.info(f"Reordered job {job_id[:8]} to priority {new_priority}")
    return True

JobQueue.cancel = cancel
JobQueue.reorder = reorder

### Observation

CR-6 splits the legacy `get_state()` UI-shaped dict into four typed accessors: `get_pending()`, `get_running()`, `get_history(limit)`, `get_stats()`. The legacy `get_state()` remains as a REMOVE-AFTER-OVERHAUL shim (see the shim cell below this section) so existing UI consumers continue to work until SG-48 sweeps them.

**CR-14 (stage 7)** replaced the log-retrieval surface outright: `get_job_logs` + the `_slice_log_by_job_window` timestamp-window heuristic (structurally unsound under stage-3 same-worker concurrency — interleaved windows) are DELETED; `get_job_diagnostics(job_id)` reads EXACT job-stamped records from the diagnostics store, and `get_history_from_journal(limit)` rehydrates the durable job history from terminal STATE_TRANSITION journal rows (the `_history` migration rider — restart-surviving, unlimited depth).

In [ ]:
#| export
def get_job(
    self,
    job_id: str  # Job to retrieve
) -> Optional[Job]:  # Job or None
    """Get a job by ID."""
    return self._jobs.get(job_id)

JobQueue.get_job = get_job

In [ ]:
#| export
async def wait_for_job(
    self,
    job_id: str,  # Job to wait for
    timeout: Optional[float] = None  # Max seconds to wait
) -> Job:  # Completed/failed/cancelled job
    """Wait for a job to complete.

    Independent of the CR-6 event bus — uses a per-job `asyncio.Event` for
    the simple block-until-done affordance. Streaming consumers should use
    `events(job_id)` instead.
    """
    event = self._job_completed_events.get(job_id)
    if not event:
        job = self._jobs.get(job_id)
        if job and job.status in (JobStatus.completed, JobStatus.failed, JobStatus.cancelled):
            return job
        raise ValueError(f"Unknown job: {job_id}")

    try:
        await asyncio.wait_for(event.wait(), timeout=timeout)
    except asyncio.TimeoutError:
        raise TimeoutError(f"Timeout waiting for job {job_id}")

    return self._jobs[job_id]

JobQueue.wait_for_job = wait_for_job

In [ ]:
#| export
def get_pending(self) -> List[Job]:  # Pending jobs, priority-sorted
    """Get pending jobs, priority-sorted (higher priority first, then FIFO)."""
    return sorted(self._pending)

JobQueue.get_pending = get_pending

In [ ]:
#| export
def get_running_jobs(self) -> List[Job]:  # All currently-executing jobs
    """All in-flight jobs (stage 3: the queue is multi-lane)."""
    return list(self._running.values())


def get_running(self) -> Optional[Job]:  # Running job or None
    """First currently-executing job (earliest started), or None if idle.

    REMOVE-AFTER-OVERHAUL: pre-stage-3 single-lane affordance kept for the
    legacy `get_state` shim + existing single-lane consumers; multi-lane
    callers use `get_running_jobs()`. The stage-9 consumer cascade migrates
    the remaining call sites.
    """
    if not self._running:
        return None
    return min(self._running.values(),
               key=lambda j: j.started_at or j.submitted_at)

JobQueue.get_running_jobs = get_running_jobs
JobQueue.get_running = get_running

In [ ]:
#| export
def get_history(
    self,
    limit: Optional[int] = None,  # Max jobs to return (most recent N); None = all
) -> List[Job]:  # Completed/failed/cancelled jobs, most recent first
    """Get completed jobs, most recent first.

    If `limit` is provided, returns the most recent N. The internal history
    list grows append-only up to `max_history`, so older jobs are evicted
    in submission order (oldest first).
    """
    history = self._history if limit is None else self._history[-limit:]
    return list(reversed(history))

JobQueue.get_history = get_history

In [ ]:
#| export
def get_stats(self) -> QueueStats:  # Aggregate counts
    """Get aggregate queue stats — total counts by terminal status."""
    return QueueStats(
        total_pending=len(self._pending),
        total_completed=sum(1 for j in self._history if j.status == JobStatus.completed),
        total_failed=sum(1 for j in self._history if j.status == JobStatus.failed),
        total_cancelled=sum(1 for j in self._history if j.status == JobStatus.cancelled),
    )

JobQueue.get_stats = get_stats

In [ ]:
#| export
def get_job_diagnostics(
    self,
    job_id: str,  # Job whose diagnostic records to read
    limit: Optional[int] = 200,  # Max records (None = all)
    after_seq: Optional[int] = None,  # Tail cursor for follow-style reads
) -> List[DiagnosticRecord]:  # Job-stamped records, oldest first
    """EXACT per-job diagnostics (CR-14; replaces `get_job_logs`).

    Records were stamped with the job id IN THE WORKER via the call-envelope
    contextvar — no timestamp windows, no over-fetch, correct under stage-3
    same-worker concurrency and across multi-instance plugins (both of which
    the deleted `_slice_log_by_job_window` heuristic got wrong). Follow-style
    consumers poll with `after_seq` (the LOG_APPENDED replacement).

    Returns [] when no diagnostics store is configured.
    """
    if self._diagnostics is None:
        return []
    return self._diagnostics.query_records(
        job_id=job_id, limit=limit, after_seq=after_seq)


def get_history_from_journal(
    self,
    limit: Optional[int] = None,  # Most recent N terminal jobs (None = all)
) -> List[Job]:  # Rehydrated job records, most recent first
    """Durable job history (the CR-14 `_history` migration rider).

    Rehydrates Job records from terminal STATE_TRANSITION journal rows —
    restart-surviving and unbounded, unlike the in-memory `get_history`
    working set (`max_history` eviction). Rehydrated Jobs are RECORDS:
    args/kwargs/result are not journaled (results live in capability DBs;
    parameters in run manifests) — identity, timing, status, error,
    composition/task fields are present.

    Falls back to the in-memory history when no journal is configured.
    """
    if self._journal is None:
        return self.get_history(limit)
    out: List[Job] = []
    for ev in self._journal.terminal_state_events(limit=limit):
        snap = ev.payload.get("job_snapshot")
        if snap:
            out.append(_job_from_snapshot(snap))
    return out

JobQueue.get_job_diagnostics = get_job_diagnostics
JobQueue.get_history_from_journal = get_history_from_journal

### Event Bus

The CR-6 push-based event surface. Three subscriber views (`events(job_id)`, `events_for_composition(comp_id)`, `all_events()`) all share one fan-out path: each subscriber owns a bounded `asyncio.Queue`, and the publisher routes every event to all matching subscriber keys (`"all"`, `"job:<id>"`, `"comp:<id>"`). Slow subscribers backpressure themselves — the publisher never blocks. Composition events are *tagged* job events (each carries optional `composition_id` + `node_id`) rather than a separate topic, so a single subscription captures the full per-composition narrative including the `COMPOSITION_ADVANCED` aggregate signal.

In [ ]:
#| export
def _subscriber_keys_for(event: JobEvent) -> List[str]:
    """Return the subscriber keys an event should fan out to (CR-6 / stage 3).

    Every event reaches "all" subscribers + the per-job subscribers.
    Composition-tagged events additionally reach per-composition subscribers.
    """
    keys = ["all", f"job:{event.job_id}"]
    if event.composition_id is not None:
        keys.append(f"comp:{event.composition_id}")
    return keys


def _journal_append_guarded(
    self,
    event: JournalEvent,  # Pre-built journal event (caller fills identity/payload)
) -> None:
    """Append to the journal under the wedge-gate failure contract (CR-14).

    The ONE place the named-tension resolution lives: an append failure logs
    at ERROR and wedges the queue (new submissions refuse via
    `_check_journal_wedge`) instead of raising into dispatch/finalization
    paths — raising there would leak lanes and corrupt in-flight state,
    while continuing silently would drop the audit trail. No-op without a
    configured journal. Used by `_publish_event` (job events) and by the
    direct substrate-event emissions (ADMISSION_DECIDED).
    """
    if self._journal is None:
        return
    try:
        self._journal.append(event)
    except Exception:
        self._journal_wedged = True
        self.logger.error(
            "JOURNAL APPEND FAILED for %s on job %s — queue WEDGED: new "
            "submissions will refuse until the journal is writable again.",
            event.event_type, event.job_id, exc_info=True)


def _publish_event(
    self,
    event: JobEvent,  # Event to emit
) -> None:
    """The SINGLE emission path (CR-14: journal-primary).

    Class routing at the one place every event passes through:
    - journal-class events (everything except `LIVENESS_EVENT_TYPES`)
      become durable journal rows FIRST, then fan out to live subscribers —
      emitting IS writing the record; the bus is a live tail of the journal.
    - liveness-class events (PROGRESS_CHANGED / RESOURCE_SNAPSHOT) fan out
      only; their final values ride the terminal STATE_TRANSITION row.

    Journal failures follow the wedge-gate contract — see
    `_journal_append_guarded`.

    Fan-out: slow subscribers backpressure themselves via `asyncio.QueueFull`
    drop — publisher never blocks. Each subscriber tracks `dropped_count` so
    operators / future telemetry can surface backpressure visibility.
    """
    if event.type.value not in LIVENESS_EVENT_TYPES:
        self._journal_append_guarded(JournalEvent(
            event_type=event.type.value,
            ts=event.timestamp,
            run_id=event.run_id,
            job_id=event.job_id,
            composition_id=event.composition_id,
            node_id=event.node_id,
            capability_instance_id=event.capability_instance_id,
            actor=event.actor,
            payload=event.payload,
        ))

    for key in _subscriber_keys_for(event):
        # `self._subscribers` is a defaultdict but we use `.get` to avoid
        # accidentally creating empty lists for unused keys (which would
        # otherwise accumulate).
        for sub in self._subscribers.get(key, []):
            try:
                sub.queue.put_nowait(event)
            except asyncio.QueueFull:
                sub.dropped_count += 1
                self.logger.warning(
                    f"Subscriber to '{key}' dropped event "
                    f"(dropped_count={sub.dropped_count}); subscriber is "
                    f"falling behind."
                )


async def _subscribe(
    self,
    key: str,  # Subscriber key ("all" | "job:<id>" | "comp:<id>")
) -> AsyncIterator[JobEvent]:
    """Internal: register a subscription; yield events until the consumer
    exits the async generator.

    Cleanup runs in `finally` so the subscriber is unregistered even if the
    consumer raises or is cancelled. Empty key lists are deleted to avoid
    memory accumulation across many short-lived subscriptions.
    """
    sub = _Subscription(queue=asyncio.Queue(maxsize=self.SUBSCRIBER_QUEUE_MAXSIZE))
    self._subscribers[key].append(sub)
    try:
        while True:
            event = await sub.queue.get()
            yield event
    finally:
        try:
            self._subscribers[key].remove(sub)
            if not self._subscribers[key]:
                del self._subscribers[key]
        except (ValueError, KeyError):
            # Already removed (e.g., queue stop drained); idempotent cleanup.
            pass


async def events(
    self,
    job_id: str,  # Filter to events for this job
) -> AsyncIterator[JobEvent]:
    """Subscribe to events for a single job (async generator).

    Yields events as they fire. Multiple concurrent subscribers to the same
    job_id each get their own independent stream — useful for multi-tab UIs.
    Late subscribers catch up exactly via the journal: query
    `journal.query(job_id=..., after_seq=cursor)` then follow live — the
    bus is a tail of the journal, not the record itself (CR-14).
    """
    async for evt in self._subscribe(f"job:{job_id}"):
        yield evt


async def events_for_composition(
    self,
    composition_id: str,  # Filter to events tagged with this composition
) -> AsyncIterator[JobEvent]:
    """Subscribe to events for all member jobs of a composition (async
    generator).

    Yields the unified per-composition narrative: member-job lifecycle events
    interleaved with `COMPOSITION_ADVANCED` aggregate signals (stage 3).
    """
    async for evt in self._subscribe(f"comp:{composition_id}"):
        yield evt


async def all_events(self) -> AsyncIterator[JobEvent]:
    """Subscribe to all events (firehose; async generator).

    Useful for global dashboards, audit logs, and telemetry sinks that need
    the complete event stream rather than a filtered view.
    """
    async for evt in self._subscribe("all"):
        yield evt


JobQueue._journal_append_guarded = _journal_append_guarded
JobQueue._publish_event = _publish_event
JobQueue._subscribe = _subscribe
JobQueue.events = events
JobQueue.events_for_composition = events_for_composition
JobQueue.all_events = all_events

### Composition Submission & Control

Stage 3 (CR-16): compositions — DAGs of capability-invocation nodes with
execution-time-bound inputs — REPLACE the pre-bound `submit_sequence` batch
model outright (the pass-2 Thread-4 decision; the 2026-05-31 opt-in framing
was superseded). The data model, validation, and binding resolution live in
`core/ports.ipynb`; the queue owns submission, lazy member-Job creation,
advancement, cancellation, and finalization.

In [ ]:
#| export
async def submit_composition(
    self,
    comp: Composition,  # Composition to run (validated at submit)
) -> str:  # composition run id
    """Submit a composition — a DAG of capability-invocation nodes with
    execution-time-bound inputs (stage 3; replaces `submit_sequence`).

    Validates upfront: structural validation via `new_composition_run`
    (duplicate ids / unknown refs / cycles → `CompositionValidationError`)
    and the disabled-plugin gate across all nodes (`CapabilityDisabledError`),
    matching the sequence-era precedent. Member Jobs are created LAZILY —
    only dependency-free nodes have Jobs at submit; downstream nodes get
    their kwargs materialized from upstream results at advancement time.

    CR-14: refuses loudly when the journal is wedged (the same gate as
    `submit`).

    Consumers wait via `wait_for_composition`, observe via
    `events_for_composition`, inspect via `get_composition`.
    """
    from cjm_plugin_system.core.errors import CapabilityDisabledError

    self._check_journal_wedge()

    for n in comp.nodes:
        meta = self._deps.get_capability_meta(n.capability_instance_id)
        if meta is not None and not meta.enabled:
            self.logger.error(
                f"Composition submission rejected: node {n.id!r} targets "
                f"disabled plugin {n.capability_instance_id}")
            raise CapabilityDisabledError(n.capability_instance_id)

    run = new_composition_run(comp, str(uuid.uuid4()))
    self._compositions[run.id] = run
    self._composition_completed_events[run.id] = asyncio.Event()

    # Empty composition — completed-at-submit so the API is total (mirrors
    # the empty-sequence precedent).
    if not comp.nodes:
        run.status = NodeState.completed
        run.completed_at = datetime.now(timezone.utc)
        self._composition_completed_events[run.id].set()
        self.logger.info(
            f"Submitted empty composition {run.id[:8]} (immediately completed)")
        return run.id

    started = await self._start_ready_nodes(run)
    self.logger.info(
        f"Submitted composition {run.id[:8]} ({len(comp.nodes)} nodes, "
        f"fail_fast={comp.fail_fast}, priority={comp.priority}; "
        f"{len(started)} initially ready)")
    return run.id

JobQueue.submit_composition = submit_composition

In [ ]:
#| export
async def wait_for_composition(
    self,
    composition_id: str,  # Composition to wait for
    timeout: Optional[float] = None,  # Max seconds to wait
) -> CompositionRun:  # Terminal run record
    """Block until a composition reaches terminal status (the
    `wait_for_job` analog for compositions)."""
    event = self._composition_completed_events.get(composition_id)
    if event is None:
        run = self._compositions.get(composition_id)
        if run is not None and run.status != NodeState.running:
            return run
        raise ValueError(f"Unknown composition: {composition_id}")
    try:
        await asyncio.wait_for(event.wait(), timeout=timeout)
    except asyncio.TimeoutError:
        raise TimeoutError(f"Timeout waiting for composition {composition_id}")
    return self._compositions[composition_id]

JobQueue.wait_for_composition = wait_for_composition

In [ ]:
#| export
async def cancel_composition(
    self,
    composition_id: str  # Composition to cancel
) -> bool:  # True if cancellation was recorded
    """Cancel an in-flight composition (USER intent — the run lands
    `cancelled`).

    Records intent FIRST (`cancel_requested`) so member-cancel callbacks
    racing through `_advance_composition` derive the right terminal status;
    then marks never-started nodes cancelled and cancels every member whose
    Job is still pending or running (in-flight members resolve through the
    per-job cooperative-cancel machinery and finalize the run on the way
    out). Returns False if the composition is unknown or already terminal.
    """
    run = self._compositions.get(composition_id)
    if run is None or run.status != NodeState.running:
        return False
    run.cancel_requested = True
    self.logger.info(f"Cancellation requested for composition {composition_id[:8]}")

    # Nodes that never got a Job can be marked directly.
    for nr in run.node_runs.values():
        if nr.state == NodeState.pending:
            nr.state = NodeState.cancelled

    # Members with live Jobs go through the per-job cancel path (each
    # resolution re-enters _advance_composition and finalization).
    for nr in list(run.node_runs.values()):
        if nr.state == NodeState.running and nr.job_id:
            await self.cancel(nr.job_id)

    self._maybe_finalize_composition(run)
    return True

JobQueue.cancel_composition = cancel_composition

In [ ]:
#| export
def get_composition(
    self,
    composition_id: str  # Composition to retrieve
) -> Optional[CompositionRun]:  # CompositionRun or None
    """Get a composition run by id (read-only inspection)."""
    return self._compositions.get(composition_id)

JobQueue.get_composition = get_composition

In [ ]:
#| export
async def _start_ready_nodes(
    self,
    run: CompositionRun,  # Composition being advanced
) -> List[str]:  # Node ids whose member Jobs were created + enqueued
    """Create + enqueue member Jobs for every currently-ready node (stage 3).

    This is where execution-time binding happens: each ready node's kwargs
    are materialized from upstream results via `resolve_node_kwargs`. A
    binding failure is recorded as that NODE's failure (dependents skip,
    fail-fast housekeeping applies) rather than raising to the caller — by
    the time bindings resolve, the composition is mid-flight and the failure
    must flow through the same path as a member-job failure.

    CR-14 follow-up: member Jobs inherit the composition's `run_id`/`actor`
    correlation tags so every member's journal rows link to the host run.
    """
    started: List[str] = []
    for nid in run.ready_nodes():
        node = run.nodes_by_id[nid]
        try:
            kwargs = resolve_node_kwargs(node, run.results_by_node())
        except CompositionBindingError as e:
            self.logger.error(
                f"Composition {run.id[:8]} node {nid!r} binding failed: {e}")
            run.record_result(
                nid, NodeState.failed,
                error=map_bare_exception_to_job_error(
                    e, capability_instance_id=node.capability_instance_id))
            run.skip_dependents(nid)
            if run.composition.fail_fast and not run.cancel_requested:
                await self._cancel_pending_members(run)
            continue
        member = Job(
            id=str(uuid.uuid4()),
            capability_instance_id=node.capability_instance_id,
            args=(),
            kwargs=kwargs,
            priority=node.priority if node.priority else run.composition.priority,
            composition_id=run.id,
            node_id=nid,
            task_name=node.task_name,
            method=node.method,
            run_id=(run.composition.run_id if run.composition.run_id is not None
                    else self.default_run_id),
            actor=(run.composition.actor if run.composition.actor is not None
                   else self.default_actor),
            control=node.control,
        )
        run.record_started(nid, member.id)
        await self._enqueue_job(member)
        started.append(nid)
    return started

JobQueue._start_ready_nodes = _start_ready_nodes

In [ ]:
#| export
async def _advance_composition(
    self,
    completed_job: Job  # Member job that just reached terminal status
) -> None:
    """Advance a composition after a member job completes (stage 3).

    Records the member outcome; on success enqueues newly-ready downstream
    nodes (emitting COMPOSITION_ADVANCED); on failure/cancellation skips
    transitive dependents and (fail_fast) cancels independent pending
    members. Finalizes the run when every node is terminal. Always called
    OUTSIDE the queue's main lock — may take it via `_enqueue_job`.
    """
    run = self._compositions.get(completed_job.composition_id)
    if run is None:
        return

    # completed/failed/cancelled JobStatus values map 1:1 onto NodeState.
    state = NodeState(completed_job.status.value)
    run.record_result(
        completed_job.node_id, state,
        result=completed_job.result if state == NodeState.completed else None,
        error=completed_job.error,
    )

    if state == NodeState.completed:
        started = await self._start_ready_nodes(run)
        if started:
            self._publish_event(JobEvent(
                type=JobEventType.COMPOSITION_ADVANCED,
                job_id=completed_job.id,
                capability_instance_id=completed_job.capability_instance_id,
                composition_id=run.id,
                node_id=completed_job.node_id,
                run_id=completed_job.run_id,
                actor=completed_job.actor,
                payload={
                    "completed_node": completed_job.node_id,
                    "enqueued_nodes": started,
                },
            ))
    else:
        skipped = run.skip_dependents(completed_job.node_id)
        if skipped:
            self.logger.info(
                f"Composition {run.id[:8]}: node {completed_job.node_id!r} "
                f"{state.value}; skipped dependents {skipped}")
        if run.composition.fail_fast and not run.cancel_requested:
            await self._cancel_pending_members(run)

    self._maybe_finalize_composition(run)

JobQueue._advance_composition = _advance_composition

In [ ]:
#| export
async def _cancel_pending_members(
    self,
    run: CompositionRun,  # Composition whose pending members should stop
) -> None:
    """Fail-fast housekeeping: stop members that have not actually started
    executing (stage 3, ratified failure semantics).

    IN-FLIGHT members run to completion — their results and caches are kept,
    and force-killing a half-done GPU job buys nothing. Nodes that never got
    a Job are marked cancelled directly; members whose Jobs still sit in the
    pending heap go through the per-job cancel path (which re-enters
    `_advance_composition` for each, transitioning them to terminal).
    """
    for nr in run.node_runs.values():
        if nr.state == NodeState.pending:
            nr.state = NodeState.cancelled
    for nr in list(run.node_runs.values()):
        if nr.state == NodeState.running and nr.job_id:
            job = self._jobs.get(nr.job_id)
            if job is not None and job.status == JobStatus.pending:
                await self.cancel(nr.job_id)

JobQueue._cancel_pending_members = _cancel_pending_members

In [ ]:
#| export
def _maybe_finalize_composition(
    self,
    run: CompositionRun,  # Composition to check for terminal state
) -> None:
    """Finalize the run once every node is terminal (idempotent; stage 3)."""
    if run.status != NodeState.running or not run.all_terminal():
        return
    run.status = run.derive_terminal_status()
    run.completed_at = datetime.now(timezone.utc)
    event = self._composition_completed_events.get(run.id)
    if event is not None:
        event.set()
    succeeded = sum(1 for nr in run.node_runs.values()
                    if nr.state == NodeState.completed)
    self.logger.info(
        f"Composition {run.id[:8]} {run.status.value} "
        f"({succeeded}/{len(run.node_runs)} nodes completed)")

JobQueue._maybe_finalize_composition = _maybe_finalize_composition

### Resource Snapshots

`get_resource_snapshot(job_id)` returns a point-in-time `ResourceSnapshot` for a job, composing worker proxy `get_stats()` with the configured sysmon plugin's CR-3 typed `list_processes()` (per-PID matching) and `get_system_status()` (global GPU stats). `_sample_resource_snapshot` is the internal helper invoked both from this method and from `_poll_progress` (which emits `RESOURCE_SNAPSHOT` events at `resource_snapshot_cadence_polls` cadence). All sysmon calls are best-effort — failures are swallowed and the snapshot returns with worker stats only.

In [ ]:
#| export
def _sample_resource_snapshot(
    self,
    job: Job  # Job to sample resources for
) -> Optional[ResourceSnapshot]:
    """Sample worker + sysmon stats for a job (CR-6 Stage 3 internal helper).

    Returns None if the worker proxy doesn't support `get_stats` or the call
    fails — substrate can't fabricate a snapshot. Sysmon enrichment is
    best-effort: if the named plugin isn't loaded / errors / lacks the CR-3
    typed methods, GPU fields stay None and the worker-only snapshot is
    returned.

    Subprocess-spawning plugins (e.g. Voxtral-vLLM's managed vLLM server)
    spawn grandchild PIDs that hold GPU memory the worker itself doesn't.
    GPU attribution delegates to `attribute_gpu_to_worker_subtree`, which
    intersects the worker-reported `subtree_pids` set with sysmon's per-PID
    GPU enumeration. The pre-fix path matched only `worker_pid` and reported
    `gpu_memory_mb=None` for any subprocess-spawning plugin.
    """
    proxy = self._deps.get_capability(job.capability_instance_id)
    if not proxy or not hasattr(proxy, 'get_stats'):
        return None

    try:
        stats = proxy.get_stats()
    except Exception:
        return None

    snapshot = ResourceSnapshot(
        timestamp=datetime.now(timezone.utc),
        worker_pid=stats.get('pid', 0),
        cpu_percent=stats.get('cpu_percent', 0.0),
        memory_rss_mb=stats.get('memory_rss_mb', 0.0),
    )

    # Best-effort sysmon enrichment via CR-3 typed methods.
    if self._sysmon_name:
        sysmon = self._deps.get_capability(self._sysmon_name)
        if sysmon is not None:
            # Per-subtree GPU usage via the shared substrate helper. Returns None
            # when sysmon is unreachable; substrate leaves GPU fields at their
            # defaults. Returns a dict with `gpu_memory_mb` and `gpu_index` when
            # sysmon is reachable (the dict's gpu_memory_mb is 0.0 for CPU-only
            # plugins on a GPU box — honest "no GPU usage" signal).
            attribution = attribute_gpu_to_worker_subtree(stats, sysmon)
            if attribution is not None:
                snapshot.gpu_memory_mb = attribution.get('gpu_memory_mb')
                snapshot.gpu_index = attribution.get('gpu_index')

            # Global GPU stats via get_system_status()
            try:
                sys_stats = sysmon.get_system_status() if hasattr(sysmon, 'get_system_status') else None
                if sys_stats is not None:
                    snapshot.gpu_type = (
                        sys_stats.get('gpu_type') if isinstance(sys_stats, dict)
                        else getattr(sys_stats, 'gpu_type', None)
                    )
                    snapshot.gpu_total_mb = (
                        sys_stats.get('gpu_total_memory_mb') if isinstance(sys_stats, dict)
                        else getattr(sys_stats, 'gpu_total_memory_mb', None)
                    )
                    snapshot.gpu_load_percent = (
                        sys_stats.get('gpu_load_percent') if isinstance(sys_stats, dict)
                        else getattr(sys_stats, 'gpu_load_percent', None)
                    )
            except Exception:
                pass

    return snapshot


def get_resource_snapshot(
    self,
    job_id: str  # Job to sample resources for
) -> Optional[ResourceSnapshot]:
    """Get a point-in-time resource snapshot for a job (CR-6 Stage 3).

    Returns None if the job is unknown or the worker proxy doesn't expose
    `get_stats`. Composes worker stats with sysmon GPU stats when the queue
    is configured with a `sysmon_capability_name`.
    """
    job = self._jobs.get(job_id)
    if not job:
        return None
    return self._sample_resource_snapshot(job)


JobQueue._sample_resource_snapshot = _sample_resource_snapshot
JobQueue.get_resource_snapshot = get_resource_snapshot


### Lifecycle

In [ ]:
#| export
async def start(self) -> None:
    """Start the queue processor.

    CR-6 Stage 4: installs the substrate-side retry observer on `_deps`
    (typically a CapabilityManager). When CR-7's reactive-retry path fires for
    a running job, the observer updates `Job.retry_count` and publishes a
    RETRY_STARTED event tagged with the in-flight job. Previous observer
    value (if any) is saved + restored in stop() for cooperative coexistence.
    """
    if self._running_flag:
        return

    # CR-6 Stage 4: install retry observer. Uses setattr defensively so
    # non-CapabilityManager deps implementations don't fail — they just accept
    # an unused attribute. CapabilityManager's execute paths look up
    # `getattr(self, '_on_retry', None)` so the hook only fires when both
    # sides are wired.
    self._prev_deps_on_retry = getattr(self._deps, '_on_retry', None)
    try:
        self._deps._on_retry = self._on_manager_retry
    except (AttributeError, TypeError):
        # Some deps implementations may not allow setattr (frozen dataclass,
        # __slots__, etc.); RETRY_STARTED won't fire for those but the rest
        # of the queue works.
        self._prev_deps_on_retry = None

    self._running_flag = True
    self._processor_task = asyncio.create_task(self._process_loop())
    self.logger.info("Job queue started")

JobQueue.start = start

In [ ]:
#| export
async def stop(self) -> None:
    """Stop the queue processor gracefully.

    Stage 3: in-flight job tasks are detached lanes now — drain them with
    the same 5s budget as the processor task; leftovers are cancelled.

    CR-6 Stage 4: restores the previous `_on_retry` observer on deps to
    leave the manager in the state we found it (cooperative with other
    queue instances, tests, etc.).
    """
    self._running_flag = False
    self._job_available.set()  # Wake up the processor

    if self._processor_task:
        try:
            await asyncio.wait_for(self._processor_task, timeout=5.0)
        except asyncio.TimeoutError:
            self._processor_task.cancel()
        self._processor_task = None

    # Stage 3: drain in-flight lanes.
    tasks = [t for t in self._running_tasks.values() if not t.done()]
    if tasks:
        done, pending = await asyncio.wait(tasks, timeout=5.0)
        for t in pending:
            t.cancel()
    self._running_tasks.clear()

    # Restore previous retry observer (if any). Best-effort — same setattr
    # caveats as start().
    try:
        if self._prev_deps_on_retry is None:
            # Was unset before we touched it; clean up our attribute.
            if hasattr(self._deps, '_on_retry'):
                try:
                    del self._deps._on_retry
                except (AttributeError, TypeError):
                    self._deps._on_retry = None
        else:
            self._deps._on_retry = self._prev_deps_on_retry
    except (AttributeError, TypeError):
        pass

    self.logger.info("Job queue stopped")

JobQueue.stop = stop

In [ ]:
#| export
def _on_manager_retry(
    self,
    instance_id: str,  # Plugin instance whose execute is retrying
    attempt: int,      # 1-based retry number (CapabilityManager's loop var; first retry is 1)
    exception: BaseException,  # The CapabilityResourceError that triggered the retry
) -> None:
    """Substrate-side retry observer (CR-6 Stage 4; stage-3 multi-lane).

    Invoked synchronously by CapabilityManager's CR-7 retry loop just before
    each retry attempt. Updates `Job.retry_count` on the matching in-flight
    job + emits RETRY_STARTED. Best-effort: synchronous callback; emission
    failure shouldn't propagate back into the retry loop.

    `attempt` semantics: CapabilityManager's loop iterates
    `for attempt in range(max_retries + 1)`. The first iteration
    (`attempt=0`) is the original try and never invokes this callback —
    so the value PASSED here is already the 1-based retry number.

    Match logic (stage 3): scan the multi-lane `_running` dict for a job on
    the retrying instance. With the per-instance cap defaulting to 1, at
    most one in-flight job matches; if an instance opts into same-worker
    concurrency (SG-33 cap > 1), the first match is attributed — a known
    blunt edge recorded in the stage-3 ledger.
    """
    running = next((j for j in self._running.values()
                    if j.capability_instance_id == instance_id), None)
    if running is None:
        # No matching in-flight job (e.g. queue stopped or different instance).
        return

    running.retry_count = attempt
    self._publish_event(JobEvent(
        type=JobEventType.RETRY_STARTED,
        job_id=running.id,
        capability_instance_id=running.capability_instance_id,
        composition_id=running.composition_id,
        node_id=running.node_id,
        run_id=running.run_id,
        actor=running.actor,
        payload={
            "attempt": attempt,  # 1-based retry number (1=first retry, etc.)
            "exception_repr": repr(exception),
            "exception_category": getattr(exception, 'category', 'resource'),
        },
    ))

JobQueue._on_manager_retry = _on_manager_retry

### Internal Methods

In [ ]:
#| export
def _move_to_history(self, job: Job) -> None:
    """Move a job to history, maintaining max_history limit."""
    self._history.append(job)
    if len(self._history) > self.max_history:
        old_job = self._history.pop(0)
        # SG-13: set the evicted event before popping so any in-flight waiter
        # still holding the reference resolves immediately. Idempotent if the
        # event was already set by `_signal_job_completed`; protective if not.
        evicted = self._job_completed_events.pop(old_job.id, None)
        if evicted is not None:
            evicted.set()

JobQueue._move_to_history = _move_to_history

In [ ]:
#| export
def _signal_job_completed(self, job_id: str) -> None:
    """Signal that a job has completed."""
    event = self._job_completed_events.get(job_id)
    if event:
        event.set()

JobQueue._signal_job_completed = _signal_job_completed

In [ ]:
#| export
def _job_snapshot(job: Job) -> Dict[str, Any]:
    """Serialize a job's RECORD fields for the terminal journal row (CR-14).

    Deliberately excludes args/kwargs/result: results live in capability DBs,
    parameters in run manifests — the journal never duplicates what better
    homes already record (the attempted-vs-happened rule). The error rides as
    the JobError dict (failures are exactly what the journal exists to keep).
    """
    return {
        "id": job.id,
        "capability_instance_id": job.capability_instance_id,
        "status": job.status.value,
        "priority": job.priority,
        "submitted_at": job.submitted_at.isoformat() if job.submitted_at else None,
        "started_at": job.started_at.isoformat() if job.started_at else None,
        "completed_at": job.completed_at.isoformat() if job.completed_at else None,
        "retry_count": job.retry_count,
        "composition_id": job.composition_id,
        "node_id": job.node_id,
        "task_name": job.task_name,
        "method": job.method,
        "run_id": job.run_id,
        "actor": job.actor,
        "error": asdict(job.error) if job.error is not None else None,
    }


def _job_from_snapshot(snap: Dict[str, Any]) -> Job:
    """Rehydrate a Job RECORD from a terminal-row snapshot (CR-14).

    Tolerant on both directions: unknown snapshot keys are ignored; a
    JobError dict that no longer matches the current JobError fields
    degrades to None rather than failing the history read.
    """
    def _dt(v):
        try:
            return datetime.fromisoformat(v) if v else None
        except (TypeError, ValueError):
            return None

    error = None
    err_d = snap.get("error")
    if isinstance(err_d, dict):
        try:
            import dataclasses as _dc
            names = {f.name for f in _dc.fields(JobError)}
            error = JobError(**{k: v for k, v in err_d.items() if k in names})
        except Exception:
            error = None

    try:
        status = JobStatus(snap.get("status", "completed"))
    except ValueError:
        status = JobStatus.completed
    return Job(
        id=snap.get("id", ""),
        capability_instance_id=snap.get("capability_instance_id", ""),
        args=(),
        kwargs={},
        status=status,
        priority=int(snap.get("priority") or 0),
        submitted_at=_dt(snap.get("submitted_at")) or datetime.now(timezone.utc),
        started_at=_dt(snap.get("started_at")),
        completed_at=_dt(snap.get("completed_at")),
        retry_count=int(snap.get("retry_count") or 0),
        composition_id=snap.get("composition_id"),
        node_id=snap.get("node_id"),
        task_name=snap.get("task_name"),
        method=snap.get("method"),
        run_id=snap.get("run_id"),
        actor=snap.get("actor"),
        error=error,
    )

In [ ]:
#| export
def _emit_state_transition(
    self,
    job: Job,
    prev_status: JobStatus,
) -> None:
    """Emit a STATE_TRANSITION event for `job`'s most recent status change.

    Centralized so every transition site (start, completed, failed, cancelled,
    or future cancel-phase-driven transitions) carries identical tag context.

    CR-14: TERMINAL transitions carry the job snapshot in the payload — the
    durable journal row becomes the job's record of existence (the `_history`
    migration rider: `get_history_from_journal` rehydrates from these).
    """
    payload: Dict[str, Any] = {"from": prev_status.value, "to": job.status.value}
    if job.status in (JobStatus.completed, JobStatus.failed, JobStatus.cancelled):
        # The durable record must carry the full timing — terminal emits fire
        # before _execute_job's finalize section, so stamp completed_at HERE
        # (finalize only fills it when still unset).
        if job.completed_at is None:
            job.completed_at = datetime.now(timezone.utc)
        payload["job_snapshot"] = _job_snapshot(job)
    self._publish_event(JobEvent(
        type=JobEventType.STATE_TRANSITION,
        job_id=job.id,
        capability_instance_id=job.capability_instance_id,
        composition_id=job.composition_id,
        node_id=job.node_id,
        run_id=job.run_id,
        actor=job.actor,
        payload=payload,
    ))

JobQueue._emit_state_transition = _emit_state_transition

In [ ]:
#| export
def _emit_cancel_phase(
    self,
    job: Job,
    new_phase: CancelPhase,
) -> None:
    """Emit a CANCEL_PHASE_CHANGED event (CR-6 Stage 4).

    Updates `job.cancel_phase` to the new value and publishes an event with
    the prior phase + new phase in the payload. Centralized so every phase
    transition site in `_execute_with_cancellation` produces identically-shaped
    events.
    """
    prev_phase = job.cancel_phase
    job.cancel_phase = new_phase
    self._publish_event(JobEvent(
        type=JobEventType.CANCEL_PHASE_CHANGED,
        job_id=job.id,
        capability_instance_id=job.capability_instance_id,
        composition_id=job.composition_id,
        node_id=job.node_id,
        run_id=job.run_id,
        actor=job.actor,
        payload={
            "from": prev_phase.value if prev_phase is not None else None,
            "to": new_phase.value,
        },
    ))

JobQueue._emit_cancel_phase = _emit_cancel_phase

In [ ]:
#| export
def _emit_block_reason(
    self,
    job: Job,
    new_reason: Optional[str],
) -> None:
    """Emit a BLOCK_REASON_CHANGED event (CR-6 Stage 4 — reserved).

    Updates `job.block_reason` and publishes an event with the prior + new
    reason. Stage 4 ships this helper for future scheduler-integration use;
    the queue's current scheduling logic doesn't surface block reasons, so
    the helper is reserved for the eventual scheduler-coordination wiring.
    """
    prev_reason = job.block_reason
    if prev_reason == new_reason:
        return  # No-op: avoid event spam on repeated no-change calls
    job.block_reason = new_reason
    self._publish_event(JobEvent(
        type=JobEventType.BLOCK_REASON_CHANGED,
        job_id=job.id,
        capability_instance_id=job.capability_instance_id,
        composition_id=job.composition_id,
        node_id=job.node_id,
        run_id=job.run_id,
        actor=job.actor,
        payload={"from": prev_reason, "to": new_reason},
    ))

JobQueue._emit_block_reason = _emit_block_reason

### Multi-lane dispatch + resource-derived admission

Stage 3 replaced the pre-overhaul single-lane loop (one inline `_execute_job`
at a time, a single `self._running` slot) with **ready-set dispatch**: the
loop launches every job the admission ladder allows, as independent tasks.
Admission derives from the CR-7 empirical store (per `(instance_id,
config_hash)` measured peaks) + live sysmon telemetry — never from declared
metadata. A job with no empirical evidence runs EXCLUSIVE: its first run is
its measurement run, after which the store's keying graduates it
automatically. Worst case (empty store, no sysmon) degrades to exactly the
old single-lane behavior; CR-7 reactive retry backstops admission misses.

In [ ]:
#| export
async def _fetch_admission_stats(self) -> Dict[str, Any]:
    """Fetch live system telemetry for admission decisions (stage 3).

    Defensive-getattr: a deps implementation without `get_global_stats`
    (older test doubles) yields `{}` — admission then runs GPU-profiled jobs
    exclusive (no live headroom to verify against) and CPU-profiled jobs on
    lanes + instance caps alone. Failures degrade the same way.
    """
    fn = getattr(self._deps, 'get_global_stats', None)
    if not callable(fn):
        return {}
    try:
        stats = await fn()
        return stats if isinstance(stats, dict) else {}
    except Exception:
        return {}

JobQueue._fetch_admission_stats = _fetch_admission_stats

In [ ]:
#| export
def _pop_next_admissible(
    self,
    stats: Dict[str, Any],  # Live telemetry from _fetch_admission_stats (possibly {})
) -> Optional[Job]:  # The popped job, or None when nothing is dispatchable
    """Pop the highest-priority ADMISSIBLE pending job (stage 3).

    Scans pending jobs in priority order with SKIP-AHEAD: a blocked job
    (insufficient GPU headroom, instance cap reached) does not stall
    admissible jobs behind it. The admission ladder (stage-3 ledger,
    ratified 2026-06-10):

    1. **lanes** — at most `max_concurrent_lanes` in-flight jobs;
    2. **exclusivity** — a job with NO empirical profile runs ALONE: its
       first run IS its measurement run. The store's (instance_id,
       config_hash) keying graduates it automatically after one run and
       demotes it again whenever the config changes — staleness is dissolved
       by the keying, not solved by invalidation;
    3. **per-instance cap** — in-flight jobs per instance ≤ the instance's
       SG-33 `max_concurrent_requests`, DEFAULT 1 when unset (same-worker
       concurrency is opt-in per capability);
    4. **resources** — the empirical `gpu_memory_mb_peak_max` is admitted
       against BOTH a reservation ledger (sum of running GPU peaks ≤ total ×
       `gpu_headroom_fraction`; covers admitted-but-not-yet-loaded models)
       AND live free VRAM (covers resident idle models + external GPU
       users); `memory_mb_peak_max` against live `memory_available_mb`.
       Without sysmon stats, GPU-profiled jobs run exclusive.

    The manifest's `requires_gpu` is deliberately NOT consumed — whether a
    config uses the GPU is an empirical fact (`gpu_memory_mb_peak_max > 0`),
    not a declaration (ledger G2: the pre-overhaul scheduler quantity checks
    were dead code against v2 manifests). Worst case (no profiles, no
    sysmon) every job runs exclusive = exact pre-stage-3 single-lane
    behavior. CR-7 reactive retry is the documented backstop for admission
    misses.
    """
    if not self._pending:
        return None
    if len(self._running) >= self.max_concurrent_lanes:
        return None
    if self._running_exclusive:
        return None  # an exclusive (unprofiled) job owns the queue

    get_profile = getattr(self._deps, 'get_admission_profile', None)
    get_cap = getattr(self._deps, 'get_instance_concurrency_cap', None)
    gpu_free = stats.get('gpu_free_memory_mb')
    gpu_total = stats.get('gpu_total_memory_mb')
    mem_avail = stats.get('memory_available_mb')

    chosen: Optional[Job] = None
    chosen_exclusive = False
    chosen_gpu_peak = 0.0
    for job in sorted(self._pending):
        # 3. per-instance cap (default 1: same-worker concurrency is opt-in).
        cap = get_cap(job.capability_instance_id) if callable(get_cap) else None
        cap = cap if (isinstance(cap, int) and cap > 0) else 1
        inflight = sum(1 for j in self._running.values()
                       if j.capability_instance_id == job.capability_instance_id)
        if inflight >= cap:
            continue

        profile = get_profile(job.capability_instance_id) if callable(get_profile) else None
        if profile is None:
            # 2. exclusivity: no evidence → sole occupancy (measurement run).
            if self._running:
                continue
            chosen, chosen_exclusive, chosen_gpu_peak = job, True, 0.0
            break

        gpu_peak = float(profile.get('gpu_memory_mb_peak_max') or 0.0)
        mem_peak = float(profile.get('memory_mb_peak_max') or 0.0)
        if gpu_peak > 0.0 and (not gpu_total or gpu_free is None):
            # GPU-profiled but no live telemetry to verify against →
            # exclusive (conservative; the cores all run with sysmon).
            if self._running:
                continue
            chosen, chosen_exclusive, chosen_gpu_peak = job, True, gpu_peak
            break
        if gpu_peak > 0.0:
            budget = float(gpu_total) * self.gpu_headroom_fraction
            reserved = sum(self._gpu_reservations.values())
            if reserved + gpu_peak > budget:
                continue
            if gpu_peak > float(gpu_free):
                continue
        if mem_avail is not None and mem_peak > float(mem_avail):
            continue
        chosen, chosen_exclusive, chosen_gpu_peak = job, False, gpu_peak
        break

    if chosen is None:
        return None

    self._pending = [j for j in self._pending if j.id != chosen.id]
    heapq.heapify(self._pending)
    # Reserve the lane SYNCHRONOUSLY at pop time (under the caller's lock):
    # the dispatch loop can pop several jobs before any execute task gets
    # scheduled (no awaits on the fast path), so admission state updated
    # later in _execute_job would let every scan see an empty queue and
    # defeat the cap/exclusivity/ledger checks entirely.
    self._running[chosen.id] = chosen
    if chosen_exclusive:
        self._running_exclusive.add(chosen.id)
    if chosen_gpu_peak > 0.0:
        self._gpu_reservations[chosen.id] = chosen_gpu_peak
    return chosen

JobQueue._pop_next_admissible = _pop_next_admissible

In [ ]:
#| export
async def _process_loop(self) -> None:
    """Main dispatch loop (stage 3: multi-lane ready-set dispatch).

    `_job_available` means "dispatch state may have changed" — set on submit
    AND on every job completion, cleared only after a scan that dispatched
    nothing. Each pass: fetch admission telemetry (outside the lock), pop
    the highest-priority admissible job under the lock, and launch it as an
    independent task tracked in `_running_tasks` (awaited by `stop`). The
    pre-stage-3 loop executed one job at a time inline.

    CR-14 follow-up: each ADMIT is journaled as an ADMISSION_DECIDED row —
    emitted AFTER the lock releases (sqlite I/O never rides the locked fast
    path; the decision detail is recovered from the admission ledgers the
    pop updated synchronously). Blocked jobs are NOT journaled per scan —
    the scan loop re-evaluates them on every pass and would spam rows; the
    reserved BLOCK_REASON_CHANGED transition channel is the place block
    visibility lands when the scheduler-coordination wiring happens.
    """
    while self._running_flag:
        await self._job_available.wait()

        if not self._running_flag:
            break

        # Admission inputs fetched outside the lock (sysmon HTTP round-trip
        # must not block other queue operations). Slightly stale stats are
        # acceptable — the margin is blunt by design.
        stats = await self._fetch_admission_stats() if self._pending else {}

        async with self._lock:
            job = self._pop_next_admissible(stats)
            if job is None:
                # Nothing dispatchable (empty / lanes full / nothing
                # admissible): sleep until a submit or completion changes
                # the dispatch state.
                self._job_available.clear()
                continue

        task = asyncio.create_task(self._execute_job(job))
        self._running_tasks[job.id] = task

        # CR-14 follow-up: the admission decision is an account-of-action.
        # Pure journal row (no bus fan-out — it is not a JobEventType);
        # wedge-gate failure contract applies.
        self._journal_append_guarded(JournalEvent(
            event_type=SubstrateEventType.ADMISSION_DECIDED.value,
            job_id=job.id,
            run_id=job.run_id,
            composition_id=job.composition_id,
            node_id=job.node_id,
            capability_instance_id=job.capability_instance_id,
            actor=job.actor,
            payload={
                "exclusive": job.id in self._running_exclusive,
                "gpu_reserved_mb": self._gpu_reservations.get(job.id, 0.0),
                "lanes_in_use": len(self._running),
                "stats_available": bool(stats),
            },
        ))

JobQueue._process_loop = _process_loop

In [ ]:
#| export
async def _execute_job(self, job: Job) -> None:
    """Execute a single job (runs as an independent task per lane; stage 3)."""
    self.logger.info(f"Starting job {job.id[:8]} ({job.capability_instance_id})")

    # Mark as running + emit transition pending → running
    prev_status = job.status
    job.status = JobStatus.running
    job.started_at = datetime.now(timezone.utc)
    # Lane already reserved at pop time (_pop_next_admissible) — see the
    # admission-state synchronicity note there.
    self._emit_state_transition(job, prev_status)

    try:
        # Get the plugin proxy
        plugin = self._deps.get_capability(job.capability_instance_id)
        if not plugin:
            raise ValueError(f"Plugin not loaded: {job.capability_instance_id}")

        # Start progress polling task
        progress_task = asyncio.create_task(
            self._poll_progress(job, plugin)
        )

        # Execute with cancellation support
        try:
            result = await self._execute_with_cancellation(job, plugin)

            if job.id in self._cancel_requested:
                prev = job.status
                job.status = JobStatus.cancelled
                self._cancel_requested.discard(job.id)
                self._emit_state_transition(job, prev)
            else:
                prev = job.status
                job.status = JobStatus.completed
                job.result = result
                self._emit_state_transition(job, prev)

        except asyncio.CancelledError:
            prev = job.status
            job.status = JobStatus.cancelled
            self._emit_state_transition(job, prev)
        except Exception as e:
            prev = job.status
            job.status = JobStatus.failed
            # CR-5: typed JobError carries category + retriable + structured
            # side-channels for CapabilityError subclasses.
            job.error = map_bare_exception_to_job_error(
                e,
                capability_instance_id=job.capability_instance_id,
            )
            self.logger.error(f"Job {job.id[:8]} failed: {e}")
            self._emit_state_transition(job, prev)
        finally:
            progress_task.cancel()
            try:
                await progress_task
            except asyncio.CancelledError:
                pass

    except Exception as e:
        prev = job.status
        job.status = JobStatus.failed
        job.error = map_bare_exception_to_job_error(
            e,
            capability_instance_id=job.capability_instance_id,
        )
        self.logger.error(f"Job {job.id[:8]} failed: {e}")
        self._emit_state_transition(job, prev)

    # Finalize: release the lane + admission ledgers, then wake the
    # dispatcher (a lane freed / resources released = dispatch state changed).
    if job.completed_at is None:  # terminal emit usually stamped it (CR-14 snapshot)
        job.completed_at = datetime.now(timezone.utc)
    job.progress = 1.0 if job.status == JobStatus.completed else job.progress
    self._running.pop(job.id, None)
    self._running_exclusive.discard(job.id)
    self._gpu_reservations.pop(job.id, None)
    self._running_tasks.pop(job.id, None)

    async with self._lock:
        # SG-13: signal before moving to history so this job's event is set
        # before any eviction could pop it from the dict.
        self._signal_job_completed(job.id)
        self._move_to_history(job)

    self._job_available.set()

    # Stage 3: advance the composition after lock release. _advance_composition
    # may take the lock again via _enqueue_job (downstream members); asyncio.Lock
    # is not re-entrant, so this must happen outside the lock block above.
    if job.composition_id is not None:
        await self._advance_composition(job)

    self.logger.info(f"Job {job.id[:8]} {job.status.value}")

JobQueue._execute_job = _execute_job

In [ ]:
#| export
async def _execute_with_cancellation(
    self,
    job: Job,
    plugin: Any
) -> Any:
    """Execute job with cancellation monitoring.

    CR-6 Stage 4 wires CANCEL_PHASE_CHANGED events for the substrate's
    cooperative → force → reloading → completed state machine.

    Cooperative-success path (plugin acknowledges cancel within timeout):
        COOPERATIVE → COMPLETED
    Force-kill path (cooperative timeout):
        COOPERATIVE → FORCE → RELOADING → COMPLETED
    """
    # CR-14: set the per-call envelope around exec-task creation —
    # asyncio.create_task copies the CURRENT context, so the identity flows
    # through deps (manager) into the proxy's payload construction and onto
    # the wire, with the reset landing immediately after creation (no other
    # code in THIS task sees it). run_id/actor ride from the Job (CR-14
    # follow-up) so in-worker diagnostics carry the host-tier correlation.
    token = set_call_envelope(CallEnvelope(
        job_id=job.id,
        run_id=job.run_id,
        composition_id=job.composition_id,
        node_id=job.node_id,
        actor=job.actor,
        control=job.control or {},
    ))
    try:
        # Start execution. Task-addressed jobs (stage 4, CR-17 pt 2) route via
        # the explicit task channel; execute-channel jobs are unchanged.
        if job.task_name is not None:
            exec_task = asyncio.create_task(
                self._deps.execute_capability_task_async(
                    job.capability_instance_id, job.task_name, job.method, **job.kwargs)
            )
        else:
            exec_task = asyncio.create_task(
                self._deps.execute_capability_async(job.capability_instance_id, *job.args, **job.kwargs)
            )
    finally:
        reset_call_envelope(token)

    # Monitor for cancellation
    while not exec_task.done():
        if job.id in self._cancel_requested:
            # PHASE: COOPERATIVE — cancel signal being sent, awaiting acknowledgement
            self._emit_cancel_phase(job, CancelPhase.COOPERATIVE)
            self.logger.info(f"Attempting cooperative cancel for {job.id[:8]}")

            if hasattr(plugin, 'cancel_async'):
                await plugin.cancel_async()
            elif hasattr(plugin, 'cancel'):
                plugin.cancel()

            # Wait briefly for cooperative cancellation
            try:
                result = await asyncio.wait_for(exec_task, timeout=self.cancel_timeout)
                # Plugin acknowledged + completed within timeout.
                self._emit_cancel_phase(job, CancelPhase.COMPLETED)
                return result
            except asyncio.TimeoutError:
                # PHASE: FORCE — cooperative timeout; force-terminating worker
                self._emit_cancel_phase(job, CancelPhase.FORCE)
                self.logger.warning(f"Force terminating plugin for job {job.id[:8]}")
                exec_task.cancel()

                # PHASE: RELOADING — worker reload in progress post-force-kill
                self._emit_cancel_phase(job, CancelPhase.RELOADING)
                self._deps.reload_capability(job.capability_instance_id)

                # PHASE: COMPLETED — cancellation fully resolved
                self._emit_cancel_phase(job, CancelPhase.COMPLETED)
                raise asyncio.CancelledError()

        await asyncio.sleep(0.1)

    return exec_task.result()

JobQueue._execute_with_cancellation = _execute_with_cancellation

In [ ]:
#| export
async def _poll_progress(
    self,
    job: Job,
    plugin: Any
) -> None:
    """Poll progress + sample resources from the plugin during execution.

    Emits PROGRESS_CHANGED events when progress or status_message changes
    from the previous poll (avoids spamming the bus between meaningful
    updates). CR-6 Stage 3: also emits RESOURCE_SNAPSHOT events every
    `resource_snapshot_cadence_polls` iterations; snapshot is also stored
    on `job.last_resource_snapshot` for synchronous inspection.

    Liveness-class events (never journaled) still carry run_id/actor so
    live-tail subscribers see the same tag shape as journal-class events.
    """
    last_progress = job.progress
    last_message = job.status_message
    poll_count = 0
    while True:
        try:
            if hasattr(plugin, 'get_progress_async'):
                progress_info = await plugin.get_progress_async()
            elif hasattr(plugin, 'get_progress'):
                progress_info = plugin.get_progress()
            else:
                progress_info = None

            if progress_info is not None:
                new_progress = progress_info.get('progress', 0.0)
                new_message = progress_info.get('message', '')
                job.progress = new_progress
                job.status_message = new_message

                if new_progress != last_progress or new_message != last_message:
                    self._publish_event(JobEvent(
                        type=JobEventType.PROGRESS_CHANGED,
                        job_id=job.id,
                        capability_instance_id=job.capability_instance_id,
                        composition_id=job.composition_id,
                        node_id=job.node_id,
                        run_id=job.run_id,
                        actor=job.actor,
                        payload={"progress": new_progress, "status_message": new_message},
                    ))
                    last_progress = new_progress
                    last_message = new_message

            # CR-6 Stage 3: sample resources at cadence + emit RESOURCE_SNAPSHOT.
            poll_count += 1
            if poll_count % self.resource_snapshot_cadence_polls == 0:
                snapshot = self._sample_resource_snapshot(job)
                if snapshot is not None:
                    job.last_resource_snapshot = snapshot
                    self._publish_event(JobEvent(
                        type=JobEventType.RESOURCE_SNAPSHOT,
                        job_id=job.id,
                        capability_instance_id=job.capability_instance_id,
                        composition_id=job.composition_id,
                        node_id=job.node_id,
                        run_id=job.run_id,
                        actor=job.actor,
                        payload={"snapshot": asdict(snapshot)},
                    ))

            if progress_info is None and self.resource_snapshot_cadence_polls <= 1:
                # Pathological: no progress support AND every-poll snapshot
                # cadence — we'd spin forever without yielding. Defensive
                # break matches pre-Stage-3 behavior when no progress hook
                # exists.
                if not hasattr(plugin, 'get_stats'):
                    break

        except Exception:
            pass  # Ignore polling errors

        await asyncio.sleep(self.progress_poll_interval)

JobQueue._poll_progress = _poll_progress

In [ ]:
#| hide
# SG-13 regression: when `_move_to_history` evicts an event from
# `_job_completed_events`, the evicted event must be set so any in-flight waiter
# still holding the reference resolves immediately rather than hanging forever.
#
# Updated for CR-6: `JobQueue` now takes `deps=` instead of `manager=`; `Job`
# is constructed with `capability_instance_id=` instead of `capability_name=`.
import asyncio as _asyncio
from unittest.mock import MagicMock as _MagicMock


async def _sg13_eviction_signals_waiters_check():
    queue = JobQueue(deps=_MagicMock(), max_history=2)

    # Submit-style setup: create 3 jobs + their events without actually running them.
    jobs = []
    for i in range(3):
        j = Job(id=f"job-{i}", capability_instance_id="p", args=(), kwargs={})
        j.status = JobStatus.completed
        jobs.append(j)
        queue._jobs[j.id] = j
        queue._job_completed_events[j.id] = _asyncio.Event()

    # A waiter grabs a reference to job-0's event BEFORE eviction (the race window).
    held_ref = queue._job_completed_events["job-0"]
    assert not held_ref.is_set(), "event should start unset"

    # Move jobs 0 and 1 into history (no eviction yet at max_history=2).
    queue._move_to_history(jobs[0])
    queue._move_to_history(jobs[1])
    assert "job-0" in queue._job_completed_events

    # Moving job 2 evicts job 0; SG-13 fix must set job-0's event before dropping.
    queue._move_to_history(jobs[2])
    assert "job-0" not in queue._job_completed_events, "evicted entry should be gone"
    assert held_ref.is_set(), "SG-13: evicted event must be set so waiters resolve"

    # A waiter that awaits the held reference resolves immediately.
    await _asyncio.wait_for(held_ref.wait(), timeout=0.1)


_asyncio.run(_sg13_eviction_signals_waiters_check())
print("SG-13 eviction-signals-waiters: PASS")

In [ ]:
#| hide
# CR-6 Stage 1 verification (stage-3 refresh): event-bus fan-out,
# composition-tag routing, Protocol satisfaction, and backward-compat
# property shims. Smoke-tests the push-based surface and confirms the
# REMOVE-AFTER-OVERHAUL aliases still produce the pre-CR-6 shapes.
import asyncio as _asyncio_s1
from unittest.mock import MagicMock as _MagicMock_s1


class _FakeDeps:
    """Minimal concrete fake satisfying JobQueueDependencies.

    Used for the isinstance-against-Protocol assertion because Python 3.12+
    runtime_checkable Protocol checks use `inspect.getattr_static()` which
    bypasses MagicMock's __getattr__-driven attribute auto-creation. A real
    class with declared methods satisfies the static-attribute lookup.
    Stage 3 added the admission surface (profile / cap / global stats);
    stage 4 added the task channel (execute_capability_task_async).
    """
    def get_capability_meta(self, name_or_id): return None
    def get_capability(self, name_or_id): return None
    async def execute_capability_async(self, name_or_id, *args, **kwargs): return None
    def reload_capability(self, name_or_id): return None
    def get_admission_profile(self, name_or_id): return None
    def get_instance_concurrency_cap(self, name_or_id): return None
    async def get_global_stats(self): return {}
    async def execute_capability_task_async(self, name_or_id, task_name, method, **kwargs): return None


async def _stage1_event_bus_check():
    # JobQueue still accepts MagicMock as `deps` because the Protocol is
    # used for type-hinting + the runtime_checkable affordance, not as a
    # hard gate. Production callers pass CapabilityManager (a real class).
    queue = JobQueue(deps=_MagicMock_s1(), max_history=10)

    # 1. JobQueueDependencies is @runtime_checkable. Verified with a real
    # concrete class that has the declared methods (CapabilityManager's shape).
    assert isinstance(_FakeDeps(), JobQueueDependencies), \
        "_FakeDeps with the declared methods should satisfy JobQueueDependencies"

    # 2. _subscriber_keys_for returns the right routing keys.
    plain = JobEvent(type=JobEventType.STATE_TRANSITION, job_id="X", capability_instance_id="p")
    assert _subscriber_keys_for(plain) == ["all", "job:X"]
    tagged = JobEvent(
        type=JobEventType.STATE_TRANSITION, job_id="X", capability_instance_id="p",
        composition_id="C", node_id="convert",
    )
    assert _subscriber_keys_for(tagged) == ["all", "job:X", "comp:C"]

    # 3. Multi-subscriber fan-out: 2 subscribers to same job_id both receive.
    received_a, received_b = [], []

    async def _collect(gen, sink, limit):
        async for evt in gen:
            sink.append(evt)
            if len(sink) >= limit:
                return

    job_id = "job-A"
    t_a = _asyncio_s1.create_task(_collect(queue.events(job_id), received_a, 1))
    t_b = _asyncio_s1.create_task(_collect(queue.events(job_id), received_b, 1))
    await _asyncio_s1.sleep(0.05)  # Let both subscriptions register.

    queue._publish_event(JobEvent(
        type=JobEventType.STATE_TRANSITION, job_id=job_id, capability_instance_id="p",
        payload={"from": "pending", "to": "running"},
    ))
    await _asyncio_s1.wait_for(_asyncio_s1.gather(t_a, t_b), timeout=1.0)
    assert len(received_a) == 1 and len(received_b) == 1, \
        "Both subscribers should receive the event"

    # 4. Composition-tagged event fans out to "all", "job:X", AND "comp:Y".
    received_all, received_comp, received_job = [], [], []
    t_all = _asyncio_s1.create_task(_collect(queue.all_events(), received_all, 1))
    t_comp = _asyncio_s1.create_task(
        _collect(queue.events_for_composition("comp-C"), received_comp, 1))
    t_job = _asyncio_s1.create_task(_collect(queue.events("job-B"), received_job, 1))
    await _asyncio_s1.sleep(0.05)

    queue._publish_event(JobEvent(
        type=JobEventType.COMPOSITION_ADVANCED, job_id="job-B", capability_instance_id="p",
        composition_id="comp-C", node_id="align",
    ))
    await _asyncio_s1.wait_for(_asyncio_s1.gather(t_all, t_comp, t_job), timeout=1.0)
    assert len(received_all) == 1, "all-events firehose should receive every event"
    assert len(received_comp) == 1, "composition subscriber should receive tagged event"
    assert len(received_job) == 1, "job subscriber should receive its event"

    # 5. Backward-compat property aliases on Job: capability_name + created_at.
    j = Job(id="j1", capability_instance_id="whisper", args=(), kwargs={})
    assert j.capability_name == "whisper", "Job.capability_name @property aliases capability_instance_id"
    assert isinstance(j.created_at, float), "Job.created_at @property returns a float"
    assert j.created_at == j.submitted_at.timestamp()

    # 6. Backward-compat property on JobQueue: queue.manager → _deps.
    assert queue.manager is queue._deps, "JobQueue.manager @property aliases _deps"


_asyncio_s1.run(_stage1_event_bus_check())
print("CR-6 Stage 1 event-bus + backward-compat (stage-3 refresh): PASS")

In [ ]:
#| hide
# Stage-3 verification: submit_composition end-to-end (execution-time pipe
# binding, parallel fan-in, fail_fast vs user cancel, empty composition,
# disabled gate) + the multi-lane executor's admission ladder (concurrency
# across instances, exclusive-without-profile, per-instance cap, GPU ledger).
import asyncio as _asyncio_s3
from dataclasses import dataclass as _dc_s3
from cjm_plugin_system.core.ports import CompositionNode, OutputRef


@_dc_s3
class _FakeMetaC:
    enabled: bool = True


class _FakeDepsC:
    """Driver fake with the stage-3 admission surface.

    Reactors map capability_instance_id -> callable(*args, **kwargs) -> result
    (raise to fail the job). `profiles` / `caps` / `stats` drive admission;
    concurrency high-water marks are tracked globally and per instance so
    tests can assert what actually co-ran.
    """
    def __init__(self, profiles=None, caps=None, stats=None, disabled=()):
        self._reactors = {}
        self.profiles = profiles or {}
        self.caps = caps or {}
        self.stats = stats if stats is not None else {}
        self.disabled = set(disabled)
        self.call_log = []
        self.active = 0
        self.max_active = 0
        self.active_by_instance = {}
        self.max_active_by_instance = {}

    def register(self, name, reactor):
        self._reactors[name] = reactor

    def get_capability_meta(self, name_or_id):
        return _FakeMetaC(enabled=name_or_id not in self.disabled)

    def get_capability(self, name_or_id):
        return object()  # no get_progress / get_stats → polling no-ops

    async def execute_capability_async(self, name_or_id, *args, **kwargs):
        self.call_log.append((name_or_id, args, kwargs))
        self.active += 1
        self.max_active = max(self.max_active, self.active)
        cur = self.active_by_instance.get(name_or_id, 0) + 1
        self.active_by_instance[name_or_id] = cur
        self.max_active_by_instance[name_or_id] = max(
            self.max_active_by_instance.get(name_or_id, 0), cur)
        try:
            reactor = self._reactors.get(name_or_id)
            res = reactor(*args, **kwargs) if callable(reactor) else reactor
            if _asyncio_s3.iscoroutine(res):
                res = await res
            return res
        finally:
            self.active -= 1
            self.active_by_instance[name_or_id] -= 1

    def reload_capability(self, name_or_id):
        return None

    def get_admission_profile(self, name_or_id):
        return self.profiles.get(name_or_id)

    def get_instance_concurrency_cap(self, name_or_id):
        return self.caps.get(name_or_id)

    async def get_global_stats(self):
        return self.stats


_CPU = {"gpu_memory_mb_peak_max": 0.0, "memory_mb_peak_max": 50.0, "sample_count": 5}
_STATS = {"gpu_free_memory_mb": 20000.0, "gpu_total_memory_mb": 24000.0,
          "memory_available_mb": 64000.0}


async def _comp_pipe_check():
    # --- Execution-time binding: convert -> align (the qwen3-e2e shape).
    deps = _FakeDepsC(profiles={"ffmpeg": _CPU, "qwen3": _CPU}, stats=_STATS)
    deps.register("ffmpeg", lambda **k: {"output_path": "/tmp/conv.wav",
                                         "received": dict(k)})
    deps.register("qwen3", lambda **k: {"items": [k["audio"]]})
    queue = JobQueue(deps=deps, max_history=20, progress_poll_interval=0.01)
    await queue.start()
    try:
        comp_id = await queue.submit_composition(Composition(nodes=[
            CompositionNode("convert", "ffmpeg",
                            {"action": "convert", "input_path": "/tmp/a.mp3"}),
            CompositionNode("align", "qwen3",
                            {"audio": OutputRef("convert", "output_path"),
                             "text": "hello"}),
        ]))
        run = await queue.wait_for_composition(comp_id, timeout=5.0)
        assert run.status == NodeState.completed, run.status
        results = run.results_by_node()
        # The align node's kwargs were materialized from convert's result.
        align_call = next(c for c in deps.call_log if c[0] == "qwen3")
        assert align_call[2]["audio"] == "/tmp/conv.wav"
        assert align_call[2]["text"] == "hello"
        assert results["align"]["items"] == ["/tmp/conv.wav"]
        # COMPOSITION_ADVANCED-style lazy creation: align's job did not exist
        # until convert completed (call order proves it).
        assert [c[0] for c in deps.call_log] == ["ffmpeg", "qwen3"]
    finally:
        await queue.stop()


async def _comp_parallel_and_admission_check():
    # --- Parallel fan-in (VAD ∥ FA shape) with CPU profiles: both co-run.
    deps = _FakeDepsC(profiles={"vad": _CPU, "fa": _CPU}, stats=_STATS)

    async def _slow(**k):
        await _asyncio_s3.sleep(0.15)
        return {"ok": True}
    deps.register("vad", _slow)
    deps.register("fa", _slow)
    queue = JobQueue(deps=deps, max_history=20, progress_poll_interval=0.01)
    await queue.start()
    try:
        comp_id = await queue.submit_composition(Composition(nodes=[
            CompositionNode("vad", "vad", {"media_path": "/seg.wav"}),
            CompositionNode("fa", "fa", {"audio": "/seg.wav", "text": "t"}),
        ]))
        run = await queue.wait_for_composition(comp_id, timeout=5.0)
        assert run.status == NodeState.completed
        assert deps.max_active == 2, \
            f"parallel nodes with CPU profiles should co-run; max_active={deps.max_active}"
    finally:
        await queue.stop()

    # --- Per-instance cap defaults to 1: same instance never co-runs ...
    deps2 = _FakeDepsC(profiles={"p": _CPU}, stats=_STATS)
    deps2.register("p", _slow)
    q2 = JobQueue(deps=deps2, max_history=20, progress_poll_interval=0.01)
    await q2.start()
    try:
        ids = [await q2.submit("p") for _ in range(3)]
        for jid in ids:
            await q2.wait_for_job(jid, timeout=5.0)
        assert deps2.max_active_by_instance["p"] == 1, \
            f"default per-instance cap is 1; got {deps2.max_active_by_instance}"
    finally:
        await q2.stop()

    # --- ... and opts UP via max_concurrent_requests (the ffmpeg case).
    deps3 = _FakeDepsC(profiles={"p": _CPU}, caps={"p": 3}, stats=_STATS)
    deps3.register("p", _slow)
    q3 = JobQueue(deps=deps3, max_history=20, progress_poll_interval=0.01)
    await q3.start()
    try:
        ids = [await q3.submit("p") for _ in range(3)]
        for jid in ids:
            await q3.wait_for_job(jid, timeout=5.0)
        assert deps3.max_active_by_instance["p"] >= 2, \
            f"cap=3 should allow same-instance co-run; got {deps3.max_active_by_instance}"
    finally:
        await q3.stop()

    # --- Exclusivity: a no-profile job never co-runs with anything.
    deps4 = _FakeDepsC(profiles={"prof": _CPU}, stats=_STATS)  # "noprof" has no entry
    deps4.register("prof", _slow)
    deps4.register("noprof", _slow)
    q4 = JobQueue(deps=deps4, max_history=20, progress_poll_interval=0.01)
    await q4.start()
    try:
        ids = [await q4.submit("noprof"), await q4.submit("prof"),
               await q4.submit("prof")]
        for jid in ids:
            await q4.wait_for_job(jid, timeout=5.0)
        assert deps4.max_active == 1, \
            f"no-profile job must run exclusive; max_active={deps4.max_active}"
    finally:
        await q4.stop()

    # --- GPU ledger: two 6GB-peak jobs on a 10GB budget serialize; a 6GB +
    # 2GB pair co-runs. budget = 10000 * 0.9 = 9000.
    gpu6 = {"gpu_memory_mb_peak_max": 6000.0, "memory_mb_peak_max": 100.0, "sample_count": 3}
    gpu2 = {"gpu_memory_mb_peak_max": 2000.0, "memory_mb_peak_max": 100.0, "sample_count": 3}
    gstats = {"gpu_free_memory_mb": 9500.0, "gpu_total_memory_mb": 10000.0,
              "memory_available_mb": 64000.0}
    deps5 = _FakeDepsC(profiles={"g1": gpu6, "g2": gpu6}, stats=gstats)
    deps5.register("g1", _slow)
    deps5.register("g2", _slow)
    q5 = JobQueue(deps=deps5, max_history=20, progress_poll_interval=0.01)
    await q5.start()
    try:
        ids = [await q5.submit("g1"), await q5.submit("g2")]
        for jid in ids:
            await q5.wait_for_job(jid, timeout=5.0)
        assert deps5.max_active == 1, \
            f"6GB+6GB exceeds the 9GB budget; max_active={deps5.max_active}"
    finally:
        await q5.stop()

    deps6 = _FakeDepsC(profiles={"g1": gpu6, "g2": gpu2}, stats=gstats)
    deps6.register("g1", _slow)
    deps6.register("g2", _slow)
    q6 = JobQueue(deps=deps6, max_history=20, progress_poll_interval=0.01)
    await q6.start()
    try:
        ids = [await q6.submit("g1"), await q6.submit("g2")]
        for jid in ids:
            await q6.wait_for_job(jid, timeout=5.0)
        assert deps6.max_active == 2, \
            f"6GB+2GB fits the 9GB budget; max_active={deps6.max_active}"
    finally:
        await q6.stop()


async def _comp_failure_and_cancel_check():
    # --- fail_fast: producer fails -> dependent skipped, never invoked;
    # run lands failed (housekeeping, not cancelled).
    deps = _FakeDepsC(profiles={"ffmpeg": _CPU, "qwen3": _CPU}, stats=_STATS)

    def _boom(**k):
        raise RuntimeError("conversion exploded")
    deps.register("ffmpeg", _boom)
    deps.register("qwen3", lambda **k: {"items": []})
    queue = JobQueue(deps=deps, max_history=20, progress_poll_interval=0.01)
    await queue.start()
    try:
        comp_id = await queue.submit_composition(Composition(nodes=[
            CompositionNode("convert", "ffmpeg", {"input_path": "/a.mp3"}),
            CompositionNode("align", "qwen3",
                            {"audio": OutputRef("convert", "output_path")}),
        ]))
        run = await queue.wait_for_composition(comp_id, timeout=5.0)
        assert run.status == NodeState.failed, run.status
        assert run.node_runs["convert"].state == NodeState.failed
        assert run.node_runs["convert"].error is not None
        assert run.node_runs["align"].state == NodeState.skipped
        assert not any(c[0] == "qwen3" for c in deps.call_log), \
            "skipped node must never be invoked"
    finally:
        await queue.stop()

    # --- cancel_composition before dispatch (queue not started): pending
    # member cancelled, downstream cancelled, run lands cancelled (intent).
    deps2 = _FakeDepsC(profiles={"ffmpeg": _CPU, "qwen3": _CPU}, stats=_STATS)
    q2 = JobQueue(deps=deps2, max_history=20, progress_poll_interval=0.01)
    comp_id2 = await q2.submit_composition(Composition(nodes=[
        CompositionNode("convert", "ffmpeg", {"input_path": "/a.mp3"}),
        CompositionNode("align", "qwen3",
                        {"audio": OutputRef("convert", "output_path")}),
    ]))
    assert await q2.cancel_composition(comp_id2) is True
    run2 = q2.get_composition(comp_id2)
    assert run2.status == NodeState.cancelled, run2.status
    assert run2.cancel_requested is True
    assert run2.node_runs["convert"].state == NodeState.cancelled
    assert run2.node_runs["align"].state == NodeState.cancelled
    assert await q2.cancel_composition(comp_id2) is False, "already terminal"

    # --- Empty composition: completed at submit (API totality).
    comp_id3 = await q2.submit_composition(Composition(nodes=[]))
    run3 = await q2.wait_for_composition(comp_id3, timeout=1.0)
    assert run3.status == NodeState.completed

    # --- Disabled-plugin gate at submit (sequence-era precedent).
    deps3 = _FakeDepsC(disabled={"qwen3"})
    q3 = JobQueue(deps=deps3, max_history=20)
    from cjm_plugin_system.core.errors import CapabilityDisabledError
    try:
        await q3.submit_composition(Composition(nodes=[
            CompositionNode("align", "qwen3", {"audio": "/x.wav"})]))
        raise AssertionError("expected CapabilityDisabledError")
    except CapabilityDisabledError:
        pass

    # --- Structural validation surfaces at submit.
    try:
        await q3.submit_composition(Composition(nodes=[
            CompositionNode("a", "ffmpeg", {"x": OutputRef("ghost")})]))
        raise AssertionError("expected CompositionValidationError")
    except Exception as e:
        assert "ghost" in str(e)


_asyncio_s3.run(_comp_pipe_check())
_asyncio_s3.run(_comp_parallel_and_admission_check())
_asyncio_s3.run(_comp_failure_and_cancel_check())
print("Stage-3 composition + multi-lane admission verification: PASS")

In [ ]:
#| hide
# CR-6 Stage 3 verification (stage-7 refresh): resource snapshot composition
# (worker + sysmon), RESOURCE_SNAPSHOT event emission at cadence, and the
# CR-14 journal-primary emission path (routing + terminal snapshots + exact
# per-job diagnostics + durable history + the wedge gate).
import asyncio as _asyncio_s3
import tempfile as _tempfile_s7
from pathlib import Path as _Path_s7
from datetime import datetime as _datetime_s3, timezone as _tz_s3, timedelta as _td_s3
from dataclasses import dataclass as _dc_s3

from cjm_plugin_system.core.journal_store import LocalJournalStore as _LJS_s7
from cjm_plugin_system.core.diagnostics_store import (
    LocalDiagnosticsStore as _LDS_s7, DiagnosticRecord as _DR_s7,
)


class _FakeProxyS3:
    """Worker proxy fake: returns deterministic get_stats payload."""
    def __init__(self, pid=12345, cpu=42.0, rss=512.0):
        self._stats = {'pid': pid, 'cpu_percent': cpu, 'memory_rss_mb': rss}

    def get_stats(self):
        return self._stats


class _FakeSysmonS3:
    """CR-3 typed MonitorPlugin fake: returns get_system_status + list_processes."""
    def __init__(self, worker_pid=12345):
        self._worker_pid = worker_pid

    def get_system_status(self):
        return {
            'gpu_type': 'NVIDIA',
            'gpu_total_memory_mb': 24000.0,
            'gpu_load_percent': 78.0,
        }

    def list_processes(self):
        return [
            {'pid': 99999, 'gpu_index': 0, 'gpu_memory_mb': 100.0},
            {'pid': self._worker_pid, 'gpu_index': 0, 'gpu_memory_mb': 8500.0},
        ]


class _FakeDepsS3:
    """Driver fake supporting both worker-proxy + sysmon-plugin lookups."""
    def __init__(self, worker_proxy=None, sysmon=None):
        self._worker = worker_proxy
        self._sysmon = sysmon

    def get_capability_meta(self, name_or_id):
        @_dc_s3
        class _Meta:
            enabled: bool = True
        return _Meta()

    def get_capability(self, name_or_id):
        if name_or_id == 'sysmon':
            return self._sysmon
        return self._worker

    async def execute_capability_async(self, name_or_id, *args, **kwargs):
        return f"result-{args[0] if args else ''}"

    def reload_capability(self, name_or_id):
        return None


async def _stage3_resource_check():
    # --- 1. get_resource_snapshot WITHOUT sysmon: worker stats only.
    worker = _FakeProxyS3(pid=12345)
    deps = _FakeDepsS3(worker_proxy=worker)
    queue = JobQueue(deps=deps, max_history=10)
    job = Job(id="j-1", capability_instance_id="plug-a", args=(), kwargs={})
    queue._jobs[job.id] = job
    snap = queue.get_resource_snapshot("j-1")
    assert snap is not None
    assert snap.worker_pid == 12345
    assert snap.cpu_percent == 42.0
    assert snap.memory_rss_mb == 512.0
    assert snap.gpu_index is None  # No sysmon → GPU fields stay None
    assert snap.gpu_memory_mb is None
    assert snap.gpu_total_mb is None
    assert isinstance(snap.timestamp, _datetime_s3)

    # --- 2. get_resource_snapshot WITH sysmon: GPU fields populated.
    sysmon = _FakeSysmonS3(worker_pid=12345)
    deps2 = _FakeDepsS3(worker_proxy=worker, sysmon=sysmon)
    queue2 = JobQueue(deps=deps2, max_history=10, sysmon_capability_name='sysmon')
    queue2._jobs[job.id] = job
    snap2 = queue2.get_resource_snapshot("j-1")
    assert snap2 is not None
    assert snap2.worker_pid == 12345
    assert snap2.gpu_index == 0
    assert snap2.gpu_memory_mb == 8500.0  # Matched by worker_pid (not the 99999 other process)
    assert snap2.gpu_type == 'NVIDIA'
    assert snap2.gpu_total_mb == 24000.0
    assert snap2.gpu_load_percent == 78.0

    # --- 3. get_resource_snapshot returns None when proxy lacks get_stats.
    class _NoStatsProxy:
        pass
    deps3 = _FakeDepsS3(worker_proxy=_NoStatsProxy())
    queue3 = JobQueue(deps=deps3, max_history=10)
    queue3._jobs[job.id] = job
    assert queue3.get_resource_snapshot("j-1") is None

    # --- 4. Unknown job → None.
    assert queue.get_resource_snapshot("nonexistent") is None

    # --- 5. RESOURCE_SNAPSHOT event emitted at cadence during _poll_progress.
    # Build a queue that polls fast + has cadence=1 so every iteration samples.
    class _ProgressProxy:
        def __init__(self, pid=99):
            self._stats = {'pid': pid, 'cpu_percent': 10.0, 'memory_rss_mb': 50.0}

        def get_stats(self):
            return self._stats

        def get_progress(self):
            return {'progress': 0.5, 'message': 'working'}

    deps5 = _FakeDepsS3(worker_proxy=_ProgressProxy())
    queue5 = JobQueue(
        deps=deps5, max_history=10,
        progress_poll_interval=0.01,  # Fast polling for test
        resource_snapshot_cadence_polls=1,  # Sample every poll
    )
    j5 = Job(id="j-5", capability_instance_id="plug-b", args=(), kwargs={})
    queue5._jobs[j5.id] = j5

    received_snapshots = []

    async def _collect_snapshots(gen, sink, limit):
        async for evt in gen:
            if evt.type == JobEventType.RESOURCE_SNAPSHOT:
                sink.append(evt)
                if len(sink) >= limit:
                    return

    sub = _asyncio_s3.create_task(_collect_snapshots(queue5.events("j-5"), received_snapshots, 2))
    poll_task = _asyncio_s3.create_task(queue5._poll_progress(j5, _ProgressProxy()))
    try:
        await _asyncio_s3.wait_for(sub, timeout=2.0)
    finally:
        poll_task.cancel()
        try:
            await poll_task
        except _asyncio_s3.CancelledError:
            pass

    assert len(received_snapshots) == 2, f"Expected 2 RESOURCE_SNAPSHOT events; got {len(received_snapshots)}"
    snap_payload = received_snapshots[0].payload.get("snapshot")
    assert snap_payload is not None
    assert snap_payload["worker_pid"] == 99
    assert snap_payload["cpu_percent"] == 10.0
    # Also stored on the job's last_resource_snapshot
    assert j5.last_resource_snapshot is not None
    assert j5.last_resource_snapshot.worker_pid == 99


async def _stage7_journal_primary_check():
    with _tempfile_s7.TemporaryDirectory() as td:
        journal = _LJS_s7(_Path_s7(td) / "journal.db")
        diagnostics = _LDS_s7(_Path_s7(td) / "diag.db")
        deps = _FakeDepsS3(worker_proxy=_FakeProxyS3())
        queue = JobQueue(deps=deps, max_history=10,
                         journal=journal, diagnostics=diagnostics,
                         progress_poll_interval=0.01)

        # --- 6. Journal-primary emission + class routing: a real job run
        # produces durable rows for journal-class events; liveness types
        # (PROGRESS_CHANGED / RESOURCE_SNAPSHOT) are NEVER journaled.
        await queue.start()
        try:
            jid = await queue.submit("plug-a", "hello")
            done = await queue.wait_for_job(jid, timeout=5.0)
            assert done.status == JobStatus.completed
        finally:
            await queue.stop()

        rows = journal.query(job_id=jid)
        types = [r.event_type for r in rows]
        assert types.count("state_transition") == 2, types  # running + completed
        assert all(t not in ("progress_changed", "resource_snapshot") for t in types), \
            f"liveness events must never be journaled: {types}"

        # Terminal row carries the job snapshot; rehydrated durable history
        # matches (the `_history` migration rider).
        terminal = journal.terminal_state_events()
        assert len(terminal) == 1
        snap = terminal[0].payload["job_snapshot"]
        assert snap["id"] == jid and snap["status"] == "completed"
        hist = queue.get_history_from_journal()
        assert len(hist) == 1 and hist[0].id == jid
        assert hist[0].status == JobStatus.completed
        assert hist[0].started_at is not None and hist[0].completed_at is not None

        # --- 7. EXACT per-job diagnostics (replaces the deleted
        # timestamp-window slicer) + the wedge gate.
        diagnostics.append_record(_DR_s7(message="model loading", job_id=jid,
                                         worker_session_id="ws-x"))
        diagnostics.append_record(_DR_s7(message="other job", job_id="other",
                                         worker_session_id="ws-x"))
        mine = queue.get_job_diagnostics(jid)
        assert len(mine) == 1 and mine[0].message == "model loading"

        # Wedge gate: a journal whose append raises wedges the queue; the
        # NEXT submit refuses loudly (never silent audit loss, never a raise
        # mid-finalization).
        class _WedgedJournal:
            def append(self, event):
                raise RuntimeError("disk full")
        qw = JobQueue(deps=deps, max_history=10, journal=_WedgedJournal())
        j = Job(id="jw", capability_instance_id="p", args=(), kwargs={})
        j.status = JobStatus.running
        qw._jobs[j.id] = j
        qw._emit_state_transition(j, JobStatus.pending)  # ERROR-logged, no raise
        assert qw._journal_wedged is True
        try:
            await qw.submit("p")
            raise AssertionError("wedged queue must refuse new submissions")
        except RuntimeError as e:
            assert "WEDGED" in str(e)


_asyncio_s3.run(_stage3_resource_check())
_asyncio_s3.run(_stage7_journal_primary_check())
print("CR-6 Stage 3 resource snapshots + CR-14 journal-primary emission: PASS")

In [ ]:
#| hide
# CR-14 follow-up verification: run_id/actor correlation threading
# (submit → Job → JobEvent → journal rows → terminal snapshot round-trip;
# composition members inherit from Composition.run_id/actor) and the
# ADMISSION_DECIDED journal row emitted per admit by the dispatch loop.
import asyncio as _asyncio_s7f
import tempfile as _tempfile_s7f
from pathlib import Path as _Path_s7f

from cjm_plugin_system.core.journal_store import (
    LocalJournalStore as _LJS_s7f, SubstrateEventType as _SET_s7f,
)
from cjm_plugin_system.core.ports import (
    Composition as _Comp_s7f, CompositionNode as _CompNode_s7f,
)
from cjm_plugin_system.core.wire import get_call_envelope as _gce_s7f


class _RunCorrDeps:
    """Driver fake that also captures the call envelope the queue set."""
    def __init__(self):
        self.seen_envelopes = []

    def get_capability_meta(self, name_or_id):
        return None

    def get_capability(self, name_or_id):
        return object()

    async def execute_capability_async(self, name_or_id, *args, **kwargs):
        self.seen_envelopes.append(_gce_s7f())
        return {"ok": True, "got_kwargs": dict(kwargs)}

    async def execute_capability_task_async(self, name_or_id, task_name, method, **kwargs):
        self.seen_envelopes.append(_gce_s7f())
        return {"ok": True}

    def reload_capability(self, name_or_id):
        return None


async def _run_correlation_check():
    with _tempfile_s7f.TemporaryDirectory() as td:
        journal = _LJS_s7f(_Path_s7f(td) / "journal.db")
        deps = _RunCorrDeps()
        queue = JobQueue(deps=deps, max_history=10, journal=journal,
                         progress_poll_interval=0.01)
        await queue.start()
        try:
            # --- 1. submit(run_id=, actor=): reserved kwargs never reach the
            # plugin; every journal row for the job carries both tags.
            jid = await queue.submit("plug-a", payload="x",
                                     run_id="run_test_001", actor="cli:tester")
            done = await queue.wait_for_job(jid, timeout=5.0)
            assert done.status == JobStatus.completed
            assert done.result["got_kwargs"] == {"payload": "x"}, \
                "run_id/actor must not leak into plugin kwargs"

            rows = journal.query(job_id=jid)
            assert rows, "expected journal rows for the job"
            assert all(r.run_id == "run_test_001" for r in rows), \
                [(r.event_type, r.run_id) for r in rows]
            assert all(r.actor == "cli:tester" for r in rows)

            # run_id is a first-class query filter.
            by_run = journal.query(run_id="run_test_001")
            assert {r.job_id for r in by_run} == {jid}

            # --- 2. The call envelope carried run_id/actor to the execute path.
            env = deps.seen_envelopes[0]
            assert env is not None and env.run_id == "run_test_001"
            assert env.actor == "cli:tester"

            # --- 3. ADMISSION_DECIDED: one row per admit, with decision detail.
            adm = journal.query(job_id=jid,
                                event_type=_SET_s7f.ADMISSION_DECIDED.value)
            assert len(adm) == 1, [r.event_type for r in journal.query(job_id=jid)]
            assert adm[0].payload["exclusive"] is True  # no empirical profile
            assert adm[0].run_id == "run_test_001"

            # --- 4. Terminal snapshot round-trip: rehydrated history carries
            # run_id/actor.
            hist = queue.get_history_from_journal()
            assert hist[0].run_id == "run_test_001"
            assert hist[0].actor == "cli:tester"

            # --- 5. Composition members inherit Composition.run_id/actor.
            comp_id = await queue.submit_composition(_Comp_s7f(
                nodes=[_CompNode_s7f("n1", "plug-b", {"k": "v"})],
                run_id="run_test_002", actor="cli:tester",
            ))
            run = await queue.wait_for_composition(comp_id, timeout=5.0)
            assert run.status == NodeState.completed
            member_jid = run.node_runs["n1"].job_id
            mrows = journal.query(job_id=member_jid)
            assert mrows and all(r.run_id == "run_test_002" for r in mrows), \
                [(r.event_type, r.run_id) for r in mrows]

            # --- 6. Envelope-less submits stay honestly untagged.
            jid3 = await queue.submit("plug-c")
            await queue.wait_for_job(jid3, timeout=5.0)
            rows3 = journal.query(job_id=jid3)
            assert all(r.run_id is None and r.actor is None for r in rows3)

            # --- 7. Queue-scoped run context (set_run_context): defaults
            # apply when no explicit tags are passed; explicit overrides win.
            queue.set_run_context(run_id="run_ctx_003", actor="cli:ctx")
            jid4 = await queue.submit("plug-d")
            await queue.wait_for_job(jid4, timeout=5.0)
            rows4 = journal.query(job_id=jid4)
            assert rows4 and all(r.run_id == "run_ctx_003" and r.actor == "cli:ctx"
                                 for r in rows4), [(r.event_type, r.run_id) for r in rows4]
            jid5 = await queue.submit("plug-e", run_id="run_override")
            await queue.wait_for_job(jid5, timeout=5.0)
            rows5 = journal.query(job_id=jid5)
            assert rows5 and all(r.run_id == "run_override" for r in rows5)
            queue.set_run_context()  # clear
        finally:
            await queue.stop()


_asyncio_s7f.run(_run_correlation_check())
print("CR-14 follow-up run_id/actor + ADMISSION_DECIDED: PASS")

In [ ]:
#| hide
# CR-6 Stage 4 verification: CancelPhase transitions on both cooperative-success
# and force-kill paths + RETRY_STARTED emission via the _on_retry observer.
import asyncio as _asyncio_s4
from dataclasses import dataclass as _dc_s4


class _FakeMetaS4:
    enabled = True


class _FakePluginS4:
    """Worker proxy fake with cooperative cancel + slow execute."""
    def __init__(self, ack_cooperative: bool = True, exec_delay: float = 5.0):
        self._ack = ack_cooperative
        self._exec_delay = exec_delay
        self._cancel_event = _asyncio_s4.Event()

    async def cancel_async(self):
        if self._ack:
            self._cancel_event.set()

    def get_stats(self):
        return {'pid': 1234, 'cpu_percent': 5.0, 'memory_rss_mb': 10.0}


class _FakeDepsS4:
    """Driver fake; exposes the _on_retry attribute slot used by start()/stop()
    and the queue's retry observer.
    """
    def __init__(self, plugin, ack_cooperative: bool = True, raise_resource_n_times: int = 0):
        self._plugin = plugin
        self._ack = ack_cooperative
        self._resource_raises_remaining = raise_resource_n_times
        self.reload_count = 0
        self._on_retry = None  # observer slot; queue.start() overwrites this

    def get_capability_meta(self, name_or_id):
        return _FakeMetaS4()

    def get_capability(self, name_or_id):
        return self._plugin

    async def execute_capability_async(self, name_or_id, *args, **kwargs):
        # Simulate CR-7's reactive-retry loop: raise CapabilityResourceError N times,
        # invoke _on_retry observer before each retry, eventually succeed.
        from cjm_plugin_system.core.errors import CapabilityResourceError
        for attempt in range(self._resource_raises_remaining + 1):
            if attempt > 0 and self._on_retry is not None:
                # Mimic CapabilityManager's invocation right before the retry
                self._on_retry(name_or_id, attempt, CapabilityResourceError("simulated"))
            if attempt < self._resource_raises_remaining:
                continue  # Skip to next iteration (simulating raise + catch + retry)
            # Final iteration — actually run the cooperative-cancel logic.
            if self._ack:
                done, pending = await _asyncio_s4.wait(
                    [_asyncio_s4.create_task(self._plugin._cancel_event.wait()),
                     _asyncio_s4.create_task(_asyncio_s4.sleep(self._plugin._exec_delay))],
                    return_when=_asyncio_s4.FIRST_COMPLETED,
                )
                for t in pending:
                    t.cancel()
                return "cooperative-cancel-result"
            else:
                await _asyncio_s4.sleep(self._plugin._exec_delay)
                return "should-not-reach"

    def reload_capability(self, name_or_id):
        self.reload_count += 1


async def _stage4_check():
    # --- 1. Cooperative-success path: COOPERATIVE → COMPLETED.
    plugin1 = _FakePluginS4(ack_cooperative=True, exec_delay=5.0)
    deps1 = _FakeDepsS4(plugin1, ack_cooperative=True)
    queue1 = JobQueue(deps=deps1, max_history=10, cancel_timeout=0.5, progress_poll_interval=0.05)
    await queue1.start()
    try:
        job_id = await queue1.submit("plug-x")
        # Wait until job is running
        for _ in range(40):
            if queue1.get_running() and queue1.get_running().id == job_id:
                break
            await _asyncio_s4.sleep(0.05)

        phases_received: list = []

        async def _collect_phases(gen, sink, until_completed: bool):
            async for evt in gen:
                if evt.type == JobEventType.CANCEL_PHASE_CHANGED:
                    sink.append(evt.payload.get("to"))
                    if until_completed and evt.payload.get("to") == "completed":
                        return

        collector = _asyncio_s4.create_task(_collect_phases(queue1.events(job_id), phases_received, True))
        await _asyncio_s4.sleep(0.05)
        await queue1.cancel(job_id)
        await _asyncio_s4.wait_for(collector, timeout=5.0)

        assert phases_received == ["cooperative", "completed"], \
            f"Cooperative path: expected [cooperative, completed]; got {phases_received}"
        assert deps1.reload_count == 0, "Cooperative-success path must NOT reload the worker"

        job = queue1.get_job(job_id)
        assert job.cancel_phase == CancelPhase.COMPLETED
    finally:
        await queue1.stop()

    # --- 2. Force-kill path: COOPERATIVE → FORCE → RELOADING → COMPLETED.
    plugin2 = _FakePluginS4(ack_cooperative=False, exec_delay=10.0)  # Plugin ignores cancel
    deps2 = _FakeDepsS4(plugin2, ack_cooperative=False)
    queue2 = JobQueue(deps=deps2, max_history=10, cancel_timeout=0.2, progress_poll_interval=0.05)
    await queue2.start()
    try:
        job2_id = await queue2.submit("plug-y")
        for _ in range(40):
            if queue2.get_running() and queue2.get_running().id == job2_id:
                break
            await _asyncio_s4.sleep(0.05)

        phases2: list = []
        async def _collect_phases2(gen, sink):
            async for evt in gen:
                if evt.type == JobEventType.CANCEL_PHASE_CHANGED:
                    sink.append(evt.payload.get("to"))
                    if evt.payload.get("to") == "completed":
                        return

        collector2 = _asyncio_s4.create_task(_collect_phases2(queue2.events(job2_id), phases2))
        await _asyncio_s4.sleep(0.05)
        await queue2.cancel(job2_id)
        await _asyncio_s4.wait_for(collector2, timeout=5.0)

        assert phases2 == ["cooperative", "force", "reloading", "completed"], \
            f"Force-kill path: expected full sequence; got {phases2}"
        assert deps2.reload_count == 1, f"Force-kill must reload worker once; got {deps2.reload_count}"

        job2 = queue2.get_job(job2_id)
        assert job2.cancel_phase == CancelPhase.COMPLETED
        assert job2.cancel_requested_at is not None
    finally:
        await queue2.stop()

    # --- 3. RETRY_STARTED: deps simulates CapabilityResourceError twice, observer fires.
    plugin3 = _FakePluginS4(ack_cooperative=True, exec_delay=0.1)
    deps3 = _FakeDepsS4(plugin3, ack_cooperative=True, raise_resource_n_times=2)
    queue3 = JobQueue(deps=deps3, max_history=10, progress_poll_interval=0.05)
    await queue3.start()
    try:
        retries_received: list = []
        retry_collector_done = _asyncio_s4.Event()

        async def _collect_retries(gen, sink, target_count):
            async for evt in gen:
                if evt.type == JobEventType.RETRY_STARTED:
                    sink.append(evt)
                    if len(sink) >= target_count:
                        retry_collector_done.set()
                        return

        # Subscribe to all_events so we catch retries on any job_id.
        sub3 = _asyncio_s4.create_task(_collect_retries(queue3.all_events(), retries_received, 2))
        await _asyncio_s4.sleep(0.05)  # Let subscription register

        job3_id = await queue3.submit("plug-z")
        # The simulated retries fire from inside execute_capability_async; the job
        # completes after 3 iterations (2 retries + 1 success).
        try:
            await _asyncio_s4.wait_for(retry_collector_done.wait(), timeout=5.0)
        finally:
            sub3.cancel()
            try:
                await sub3
            except _asyncio_s4.CancelledError:
                pass

        assert len(retries_received) == 2, f"Expected 2 RETRY_STARTED events; got {len(retries_received)}"
        assert retries_received[0].payload["attempt"] == 1
        assert retries_received[1].payload["attempt"] == 2
        assert retries_received[0].job_id == job3_id, "RETRY_STARTED should be tagged with the in-flight job"

        # Wait for job to finalize
        await queue3.wait_for_job(job3_id, timeout=5.0)
        final_job = queue3.get_job(job3_id)
        assert final_job.retry_count == 2, f"Job.retry_count should be 2; got {final_job.retry_count}"
        assert final_job.status == JobStatus.completed
    finally:
        await queue3.stop()

    # --- 4. _emit_block_reason helper produces BLOCK_REASON_CHANGED.
    queue4 = JobQueue(deps=_FakeDepsS4(plugin1), max_history=10)
    j4 = Job(id="j-4", capability_instance_id="plug-q", args=(), kwargs={})
    queue4._jobs[j4.id] = j4
    received_block: list = []

    async def _collect_block(gen, sink):
        async for evt in gen:
            if evt.type == JobEventType.BLOCK_REASON_CHANGED:
                sink.append(evt)
                if len(sink) >= 2:
                    return

    sub4 = _asyncio_s4.create_task(_collect_block(queue4.events("j-4"), received_block))
    await _asyncio_s4.sleep(0.02)
    queue4._emit_block_reason(j4, "Worker busy")
    queue4._emit_block_reason(j4, "Worker busy")  # No-op (same reason)
    queue4._emit_block_reason(j4, "GPU unavailable")
    await _asyncio_s4.wait_for(sub4, timeout=2.0)

    assert len(received_block) == 2, "Repeated same-reason calls should dedupe"
    assert received_block[0].payload["to"] == "Worker busy"
    assert received_block[0].payload["from"] is None
    assert received_block[1].payload["from"] == "Worker busy"
    assert received_block[1].payload["to"] == "GPU unavailable"
    assert j4.block_reason == "GPU unavailable"


_asyncio_s4.run(_stage4_check())
print("CR-6 Stage 4 cancel-phase + retry + block-reason: PASS")

## Usage Example

```python
from cjm_plugin_system.core.manager import CapabilityManager
from cjm_plugin_system.core.queue import JobQueue, JobStatus
from cjm_plugin_system.core.scheduling import QueueScheduler

# Setup
manager = CapabilityManager(scheduler=QueueScheduler())
manager.discover_manifests()
manager.load_capability(manager.get_discovered_meta("sys-mon"))
manager.register_system_monitor("sys-mon")

# Create queue
queue = JobQueue(manager)
await queue.start()

# Submit jobs
job1_id = await queue.submit("whisper", audio="/path/to/audio1.mp3")
job2_id = await queue.submit("gemini-vision", image="/path/to/image.png")
job3_id = await queue.submit("whisper", audio="/path/to/audio2.mp3", priority=10)

# Monitor queue state (typed accessors)
print(f"Running: {queue.get_running()}")
print(f"Pending: {len(queue.get_pending())} jobs")

# Cancel a job
await queue.cancel(job2_id)

# Wait for a job to complete
job1 = await queue.wait_for_job(job1_id)
if job1.status == JobStatus.completed:
    print(job1.result)

# Cleanup
await queue.stop()
manager.unload_all()
```

In [ ]:
#| export
# SG-15: curate __all__ to expose only the class + value symbols.
# The patched JobQueue methods (submit, cancel, wait_for_job, events,
# submit_composition, cancel_composition, get_resource_snapshot, etc.) remain
# accessible as JobQueue.method() — they're removed from the module's
# `from ... import *` surface because invoking them standalone fails
# (they expect `self`). nbdev's auto-`__all__` lists them as free names;
# this override runs after that assignment and wins by being last.
#
# Stage 3: the sequence types (SequenceStep / StepResult / JobSequence) are
# RETIRED; the composition types live in `cjm_plugin_system.core.ports` and
# are imported from there by consumers.
__all__ = [
    'JobStatus', 'JobEventType', 'CancelPhase',
    'Job', 'JobEvent', 'QueueStats',
    'ResourceSnapshot',
    'JobQueueDependencies', 'JobQueue',
]

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()

In [ ]:
# Task-channel routing test (stage 4, CR-17 pt 2): submit validation + the
# executor routing task-addressed jobs via execute_capability_task_async while
# execute-channel jobs stay on execute_capability_async.
import asyncio as _aio


class _TaskChannelDeps:
    def __init__(self):
        self.task_calls = []
        self.exec_calls = []

    def get_capability_meta(self, name_or_id):
        return None

    def get_capability(self, name_or_id):
        return object()  # non-None: the executor's loaded-check passes; no cancel paths used

    async def execute_capability_async(self, name_or_id, *args, **kwargs):
        self.exec_calls.append((name_or_id, args, kwargs))
        return {"channel": "execute"}

    async def execute_capability_task_async(self, name_or_id, task_name, method, **kwargs):
        self.task_calls.append((name_or_id, task_name, method, kwargs))
        return {"channel": "task"}

    def reload_capability(self, name_or_id):
        pass


async def _run_task_channel_test():
    deps = _TaskChannelDeps()
    q = JobQueue(deps)
    await q.start()
    try:
        jid = await q.submit("graph", task="graph-storage", method="query_nodes",
                             query={"type": "node_query"})
        job = await q.wait_for_job(jid, timeout=10)
        assert job.status == JobStatus.completed, job.error
        assert job.result == {"channel": "task"}
        assert deps.task_calls == [
            ("graph", "graph-storage", "query_nodes", {"query": {"type": "node_query"}})]
        # execute channel unchanged
        jid2 = await q.submit("graph", "posarg", key="val")
        job2 = await q.wait_for_job(jid2, timeout=10)
        assert job2.result == {"channel": "execute"}
        assert deps.exec_calls == [("graph", ("posarg",), {"key": "val"})]
        # validation: task without method refuses at submit
        from cjm_plugin_system.core.errors import CapabilityInputError as _PIE
        try:
            await q.submit("graph", task="graph-storage")
            raise AssertionError("should have raised")
        except _PIE:
            pass
    finally:
        await q.stop()

_aio.run(_run_task_channel_test())
